# vcc-data: dati pubblici su Google Drive, verificati

Scarica i file del catalogo **dalla macchina Colab** (non dal tuo PC) e li salva in Google Drive,
uno alla volta: download sul disco della VM, controllo di dimensione e checksum, copia su Drive,
nuovo controllo della dimensione, file `.verified.json` accanto a ogni file. Se la sessione cade,
si rilancia la stessa cella: i file giÃ  verificati si saltano.

Catalogo congelato il 2026-09-28T15:52:45Z dalle API di Figshare, Zenodo, Hugging Face e S3.

| tier | file | dimensione | licenza |
|---|---|---|---|
| `mini` | 5 | 2.69 GB | CC BY 4.0 |
| `sc_replogle` | 2 | 19.36 GB | CC BY 4.0 |
| `sc_k562_gw` | 1 | 65.83 GB | CC BY 4.0 |
| `cd4` | 1 | 44.57 GB | not stated by the source: check before any use beyond research |
| `orion_hct116` | 109 | 46.58 GB | cc-by-nc-sa-4.0 |
| `orion_hek293t` | 223 | 79.68 GB | cc-by-nc-sa-4.0 |

**Di default scarica solo `mini`.** Gli altri tier si aggiungono a `TIERS` solo dopo averli decisi:
`cd4` non dichiara una licenza, Orion Ã¨ non commerciale.

Limiti da conoscere: Google Drive accetta circa 750 GB di upload al giorno (lo script si ferma a 700);
il disco della VM deve contenere il file piÃ¹ grande (65,8 GB per `sc_k562_gw`); una sessione Colab
gratuita puÃ² chiudersi dopo alcune ore, e in quel caso si rilancia.

In [ ]:
# Impostazioni: l'unica cella da modificare
TIERS = ["mini"]                     # aggiungere: "sc_replogle", "sc_k562_gw", "cd4", "orion_hct116", "orion_hek293t"
DRIVE_DIR = "/content/drive/MyDrive/vcc-data"
LIMIT = 0                            # 0 = tutti i file dei tier scelti; un numero piccolo per una prova

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import json, os, shutil, subprocess
os.makedirs(DRIVE_DIR, exist_ok=True)
gb = lambda b: f"{b / 1e9:.1f} GB"
vm, dr = shutil.disk_usage("/content"), shutil.disk_usage("/content/drive/MyDrive")
print("VM disk free:", gb(vm.free), "| Drive free (as reported by the mount):", gb(dr.free))
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"],
                     capture_output=True, text=True).stdout or "no GPU")

## Codice
Ogni cella scrive un file in `/content/vcc-code/`. L'ultima ne controlla lo sha256 e li copia anche in `DRIVE_DIR/code/`.

In [ ]:
!mkdir -p /content/vcc-code

In [ ]:
%%writefile /content/vcc-code/drive_fetch.py
"""Fetch catalog files into Google Drive from a Colab VM, one file at a time, verified.

For each file of the selected tiers:
  1. skip it if <dest>/raw/<tier>/<file>.verified.json exists and the Drive copy has the right size;
  2. download it to the VM's local disk (resumable, retried), never straight into Drive;
  3. check size and checksum (md5 or sha256 from the host; size only for S3, where the
     sha256 is computed and recorded instead);
  4. copy it into Drive under a temporary name, rename, re-check the size on Drive;
  5. write the .verified.json sidecar, append a line to <dest>/logs/, delete the VM copy.

A failed checksum stops that file (kept on the VM as .bad) and the run continues with the next.
Nothing is ever overwritten on Drive: an existing file with a wrong size is renamed .stale.

    python drive_fetch.py --catalog catalog.json --dest /content/drive/MyDrive/vcc-data --tiers mini
"""
import argparse
import hashlib
import json
import os
import shutil
import sys
import time
import urllib.request
from pathlib import Path

DAILY_UPLOAD_GUARD = 700e9   # Google Drive accepts ~750 GB of uploads per user per day


def now():
    return time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())


def digest(path: Path, algo: str) -> str:
    h = hashlib.md5() if algo == "md5" else hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(1 << 24), b""):
            h.update(block)
    return h.hexdigest()


def download(url: str, part: Path, size: int, tries: int = 5) -> None:
    for attempt in range(1, tries + 1):
        start = part.stat().st_size if part.exists() else 0
        if start == size:
            return
        req = urllib.request.Request(url, headers={"User-Agent": "vcc-drive-fetch/0.1"})
        if start:
            req.add_header("Range", f"bytes={start}-")
        try:
            with urllib.request.urlopen(req, timeout=300) as r:
                if start and r.status != 206:
                    start = 0
                done, t0, last = start, time.time(), 0.0
                with part.open("ab" if start else "wb") as f:
                    while True:
                        block = r.read(1 << 23)
                        if not block:
                            break
                        f.write(block)
                        done += len(block)
                        if time.time() - last > 10:
                            rate = (done - start) / max(time.time() - t0, 1e-6) / 1e6
                            print(f"    {done / 1e9:7.2f} / {size / 1e9:.2f} GB  {rate:6.1f} MB/s", flush=True)
                            last = time.time()
            if part.stat().st_size == size:
                return
        except Exception as exc:  # network errors: retry from where the file stopped
            print(f"    attempt {attempt} failed: {exc!r}", flush=True)
        time.sleep(min(60, 5 * attempt))
    raise RuntimeError(f"download incomplete after {tries} attempts: {part.stat().st_size if part.exists() else 0} of {size}")


def main() -> int:
    ap = argparse.ArgumentParser()
    ap.add_argument("--catalog", required=True)
    ap.add_argument("--dest", required=True, help="Drive folder, e.g. /content/drive/MyDrive/vcc-data")
    ap.add_argument("--stage", default="/content/stage", help="local VM disk")
    ap.add_argument("--tiers", nargs="+", required=True)
    ap.add_argument("--limit", type=int, default=0, help="at most this many files (0 = all)")
    args = ap.parse_args()
    cat = json.loads(Path(args.catalog).read_text())
    unknown = set(args.tiers) - set(cat["tiers"])
    if unknown:
        print("unknown tiers:", unknown, "available:", list(cat["tiers"]))
        return 2
    todo = [e for e in cat["files"] if e["tier"] in args.tiers]
    if args.limit:
        todo = todo[: args.limit]
    dest, stage = Path(args.dest), Path(args.stage)
    stage.mkdir(parents=True, exist_ok=True)
    (dest / "logs").mkdir(parents=True, exist_ok=True)
    log = dest / "logs" / f"fetch_{now().replace(':', '')}.jsonl"
    total = sum(e["bytes"] for e in todo)
    print(f"{len(todo)} files, {total / 1e9:.2f} GB, tiers {args.tiers}")
    uploaded, failures = 0, 0
    for i, e in enumerate(todo, 1):
        folder = dest / "raw" / e["tier"]
        folder.mkdir(parents=True, exist_ok=True)
        target, sidecar = folder / e["file"], folder / (e["file"] + ".verified.json")
        print(f"[{i}/{len(todo)}] {e['tier']}/{e['file']}  {e['bytes'] / 1e9:.2f} GB", flush=True)
        if sidecar.exists() and target.exists() and target.stat().st_size == e["bytes"]:
            print("    already on Drive, verified: skip")
            continue
        if target.exists():
            target.rename(target.with_name(target.name + f".stale_{now().replace(':', '')}"))
        if uploaded + e["bytes"] > DAILY_UPLOAD_GUARD:
            print("    stop: this file would pass the daily Drive upload guard; rerun tomorrow")
            break
        free = shutil.disk_usage(stage).free
        if free < e["bytes"] + 2e9:
            print(f"    stop: {free / 1e9:.1f} GB free on the VM, file needs {e['bytes'] / 1e9:.1f} GB")
            break
        part = stage / (e["file"] + ".part")
        rec = dict(utc=now(), tier=e["tier"], file=e["file"], url=e["url"], bytes=e["bytes"], algo=e["algo"])
        try:
            download(e["url"], part, e["bytes"])
            if e["algo"] in ("md5", "sha256"):
                got = digest(part, e["algo"])
                rec["checksum"] = got
                if got != e["checksum"]:
                    part.rename(part.with_name(f"{e['file']}.bad_{now().replace(':', '')}"))
                    raise RuntimeError(f"{e['algo']} mismatch: got {got}, catalog {e['checksum']}")
            else:
                rec["sha256_computed"] = digest(part, "sha256")
            tmp = target.with_name(target.name + ".copying")
            shutil.copyfile(part, tmp)
            os.replace(tmp, target)
            if target.stat().st_size != e["bytes"]:
                raise RuntimeError(f"size on Drive {target.stat().st_size} != {e['bytes']}")
            rec.update(status="verified", source=e["source"], license=e["license"])
            sidecar.write_text(json.dumps(rec, indent=1))
            part.unlink()
            uploaded += e["bytes"]
        except Exception as exc:
            rec.update(status="failed", error=repr(exc))
            failures += 1
            print("    FAILED:", exc, flush=True)
        with log.open("a") as f:
            f.write(json.dumps(rec) + "\n")
    print(f"done: {uploaded / 1e9:.2f} GB uploaded this run, {failures} failures, log {log}")
    return 1 if failures else 0


if __name__ == "__main__":
    sys.exit(main())

In [ ]:
%%writefile /content/vcc-code/build_mini.py
"""Build the mini cross-line CRISPRi dataset from raw/ into mini/.

Five public CRISPRi Perturb-seq experiments, four cell lines, one shared essential-gene
library (plus the K562 genome-wide screen):

    K562_ess, K562_gw  -> line K562 (two experiments of the same line, never source for each other)
    RPE1, HepG2, Jurkat

Every experiment is reduced to pseudobulk: for each perturbation the mean counts per cell,
normalised to 1e4 per cell over *all* the experiment's genes, then log1p. The effect of a
knockdown is delta = log1p(cpm_pert) - log1p(cpm_ctrl) on the genes common to all five
experiments. This is a log of means, not the scorer's mean of logs: the Replogle bulk files
hold means only, and one estimator for every source keeps the lines comparable.

    python build_mini.py --inspect      # print obs/var columns of each raw file, write nothing
    python build_mini.py                # write mini/

Outputs (mini/):
    genes.csv            gene symbols of the common axis, in column order
    lines.csv            line key, context group, control cells, targets kept
    basal.npy            [L, G] float32 log1p(cpm) of the controls, library over all the experiment's genes
    basal_common.npy     [L, G] the same with the library over the common genes (model features, P4)
    effects_<key>.npz    delta [T, G] float32, targets, n_cells, kd (on-target delta), target_basal
    targets.csv          union of targets: presence per line, fold (md5 of symbol mod 5), in_axis
    manifest.json        inputs with md5, counts, overlaps, thresholds, script sha256
"""
import argparse
import hashlib
import json
import os
import time
from pathlib import Path

import anndata as ad
import numpy as np
import pandas as pd
import scipy.sparse as sp

ROOT = Path(__file__).resolve().parent
# Data folder: raw/ is read from here and mini/ written here (Colab sets it to a local VM folder).
DATA = Path(os.environ.get("VCC_MINI_DATA", ROOT))

# Column choices. Replogle bulk files have no target column: the target is the second field of
# obs `gene_transcript` (also the obs index), and every row containing "non-targeting" is a control
# guide. This is how the team's stage 98 reads the same file (scripts/98_multisource_effects.py,
# k562_table); on 28/09 the Colab inspection showed the gene_transcript index, num_cells_filtered
# and non-integer X (per-cell means) in K562_ess and K562_gw.
SPEC = {
    "K562_ess": dict(group="K562", kind="bulk", target_col="gene_transcript", ctrl="non-targeting"),
    "K562_gw": dict(group="K562", kind="bulk", target_col="gene_transcript", ctrl="non-targeting"),
    "RPE1": dict(group="RPE1", kind="bulk", target_col="gene_transcript", ctrl="non-targeting"),
    "HepG2": dict(group="HepG2", kind="singlecell", target_col="perturbation", ctrl="control"),
    "Jurkat": dict(group="Jurkat", kind="singlecell", target_col="perturbation", ctrl="control"),
}
N_CELLS_COLS = ["num_cells_filtered", "num_cells", "n_cells", "cell_count"]
SYMBOL_COLS = ["gene_name", "gene_symbol", "symbol"]
MIN_CELLS = 20       # a perturbation with fewer cells is dropped (bulk: when the count is known)
MIN_CTRL_CELLS = 500
N_FOLDS = 5


def sha256(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()


def gene_symbols(var: pd.DataFrame) -> np.ndarray:
    for c in SYMBOL_COLS:
        if c in var.columns:
            return var[c].astype(str).to_numpy()
    return var.index.astype(str).to_numpy()


def inspect(sources: list[dict]) -> None:
    for e in sources:
        path = DATA / "raw" / e["file"]
        a = ad.read_h5ad(path, backed="r")
        print(f"\n=== {e['key']}  {a.n_obs} x {a.n_vars}  X={type(a.X).__name__}")
        print("obs columns:", list(a.obs.columns))
        print(a.obs.head(3).to_string()[:1500])
        print("var columns:", list(a.var.columns), "| var index head:", list(a.var.index[:5]))
        col = SPEC[e["key"]]["target_col"]
        if col in a.obs.columns:
            vc = a.obs[col].astype(str).value_counts()
            print(f"{col}: {len(vc)} levels; top:", vc.head(5).to_dict())
            print("control label present:", SPEC[e["key"]]["ctrl"] in vc.index)
        block = a.X[: min(256, a.n_obs)]
        block = block.toarray() if sp.issparse(block) else np.asarray(block)
        print("X sample: min", block.min(), "max", block.max(),
              "integer share", float(np.mean(block == np.round(block))))


def read_bulk(path: Path, spec: dict) -> tuple[np.ndarray, list[str], np.ndarray, np.ndarray, dict]:
    """Replogle raw bulk: one row per targeted transcript, X = mean counts per cell."""
    a = ad.read_h5ad(path)
    col = spec["target_col"]
    raw = (a.obs[col] if col in a.obs.columns else a.obs.index.to_series()).astype(str).to_numpy()
    is_ntc = np.array([spec["ctrl"] in s for s in raw])
    labels = np.array([spec["ctrl"] if nt else s.split("_")[1] for s, nt in zip(raw, is_ntc)])
    assert is_ntc.sum() > 0, (path.name, "no control rows")
    ncol = next((c for c in N_CELLS_COLS if c in a.obs.columns), None)
    n = a.obs[ncol].to_numpy(float) if ncol else np.full(len(labels), np.nan)
    X = a.X.toarray() if sp.issparse(a.X) else np.asarray(a.X, dtype=np.float64)
    # Several transcripts of one gene: counts-weighted sum of the per-cell means.
    uniq, inv = np.unique(labels, return_inverse=True)
    w = np.where(np.isnan(n), 1.0, n)
    M = sp.csr_matrix((w, (inv, np.arange(len(labels)))), shape=(len(uniq), len(labels)))
    sums = np.asarray(M @ X)
    n_g = np.bincount(inv, weights=w, minlength=len(uniq))
    means = sums / n_g[:, None]
    n_out = n_g if ncol else np.full(len(uniq), np.nan)
    info = {"n_cells_column": ncol, "rows": int(a.n_obs), "genes": int(a.n_vars),
            "label_source": col if col in a.obs.columns else "obs index", "control_rows": int(is_ntc.sum())}
    return means, list(uniq), n_out, gene_symbols(a.var), info


def read_singlecell(path: Path, spec: dict, chunk: int = 4096):
    """Stream cells in row blocks; sums per perturbation through a sparse indicator."""
    a = ad.read_h5ad(path, backed="r")
    assert spec["target_col"] in a.obs.columns, (path.name, list(a.obs.columns))
    labels = a.obs[spec["target_col"]].astype(str).to_numpy()
    uniq, inv = np.unique(labels, return_inverse=True)
    sums = np.zeros((len(uniq), a.n_vars), np.float64)
    neg = 0
    nonint = 0
    for s in range(0, a.n_obs, chunk):
        X = a.X[s: s + chunk]
        X = X.toarray() if sp.issparse(X) else np.asarray(X)
        neg += int((X < 0).sum())
        if s == 0:
            nonint = float(np.mean(X != np.round(X)))
        rows = inv[s: s + chunk]
        M = sp.csr_matrix((np.ones(len(rows)), (rows, np.arange(len(rows)))), shape=(len(uniq), len(rows)))
        sums += M @ X
        print(f"\r{path.name}: {min(s + chunk, a.n_obs):,} / {a.n_obs:,} cells", end="", flush=True)
    print()
    n = np.bincount(inv, minlength=len(uniq)).astype(float)
    info = {"cells": int(a.n_obs), "genes": int(a.n_vars), "negative_entries": neg,
            "noninteger_share_first_block": nonint}
    return sums / n[:, None], list(uniq), n, gene_symbols(a.var), info


def build(sources: list[dict]) -> None:
    out = DATA / "mini"
    out.mkdir(exist_ok=True)
    per = {}
    for e in sources:
        spec = SPEC[e["key"]]
        path = DATA / "raw" / e["file"]
        reader = read_bulk if spec["kind"] == "bulk" else read_singlecell
        means, labels, n, symbols, info = reader(path, spec)
        assert spec["ctrl"] in labels, (e["key"], "control label missing")
        # Unique, non-empty symbols only: a duplicated symbol cannot be placed on one axis.
        s = pd.Series(symbols)
        keep_gene = (~s.duplicated(keep=False)) & (s.str.len() > 0) & (s != "nan")
        lib = means.sum(1, keepdims=True)                       # over ALL the experiment's genes
        logm = np.log1p(means / lib * 1e4)
        ci = labels.index(spec["ctrl"])
        info["control_cells"] = None if np.isnan(n[ci]) else float(n[ci])
        per[e["key"]] = dict(logm=logm, ctrl_means=means[ci], labels=labels, n=n, ci=ci, symbols=np.asarray(symbols),
                             keep_gene=keep_gene.to_numpy(), info=info, spec=spec, entry=e)
        print(e["key"], info)

    common = set.intersection(*[set(p["symbols"][p["keep_gene"]]) for p in per.values()])
    genes = np.array(sorted(common))
    G = len(genes)
    lines_rows, basal, basal_common = [], [], []
    targets_by_line = {}
    for key, p in per.items():
        col = {g: i for i, g in enumerate(p["symbols"]) if p["keep_gene"][i]}
        cols = np.array([col[g] for g in genes])
        base_full = p["logm"][p["ci"]]
        basal.append(base_full[cols].astype(np.float32))
        # P4: controls with the library summed over the common genes only, for the model features.
        cm = p["ctrl_means"][cols]
        basal_common.append(np.log1p(cm / cm.sum() * 1e4).astype(np.float32))
        keep = [i for i, t in enumerate(p["labels"]) if i != p["ci"]
                and (np.isnan(p["n"][i]) or p["n"][i] >= MIN_CELLS)]
        tg = [p["labels"][i] for i in keep]
        delta = (p["logm"][keep][:, cols] - base_full[cols]).astype(np.float32)
        kd = np.array([p["logm"][i, col[t]] - base_full[col[t]] if t in col else np.nan
                       for i, t in zip(keep, tg)], np.float32)
        tb = np.array([base_full[col[t]] if t in col else np.nan for t in tg], np.float32)
        np.savez_compressed(out / f"effects_{key}.npz", delta=delta, targets=np.array(tg),
                            n_cells=p["n"][keep].astype(np.float32), kd=kd, target_basal=tb)
        targets_by_line[key] = set(tg)
        ctrl = p["info"]["control_cells"]
        if p["spec"]["kind"] == "singlecell":
            assert ctrl is not None and ctrl >= MIN_CTRL_CELLS, (key, ctrl)
        lines_rows.append(dict(line=key, group=p["spec"]["group"], control_cells=ctrl,
                               targets=len(tg), targets_with_kd=int(np.isfinite(kd).sum()),
                               median_kd=float(np.nanmedian(kd))))
    np.save(out / "basal.npy", np.stack(basal))
    np.save(out / "basal_common.npy", np.stack(basal_common))
    pd.DataFrame({"gene": genes}).to_csv(out / "genes.csv", index=False)
    pd.DataFrame(lines_rows).to_csv(out / "lines.csv", index=False)

    allt = sorted(set.union(*targets_by_line.values()))
    gset = set(genes)
    tdf = pd.DataFrame({"target": allt})
    for key, ts in targets_by_line.items():
        tdf[f"in_{key}"] = tdf.target.isin(ts)
    tdf["n_groups"] = sum(
        tdf[[f"in_{k}" for k in per if per[k]["spec"]["group"] == grp]].any(axis=1).astype(int)
        for grp in sorted({p["spec"]["group"] for p in per.values()}))
    tdf["in_axis"] = tdf.target.isin(gset)
    tdf["fold"] = [int(hashlib.md5(t.encode()).hexdigest(), 16) % N_FOLDS for t in allt]
    tdf.to_csv(out / "targets.csv", index=False)

    keys = list(per)
    overlap = {a: {b: len(targets_by_line[a] & targets_by_line[b]) for b in keys} for a in keys}
    manifest = dict(
        written_utc=time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        script_sha256=sha256(Path(__file__)),
        estimator="delta = log1p(1e4*mean/lib) pert - ctrl; lib over all genes of the experiment",
        thresholds=dict(min_cells=MIN_CELLS, min_ctrl_cells=MIN_CTRL_CELLS, n_folds=N_FOLDS),
        n_genes=G, n_targets_union=len(allt),
        targets_in_all_groups=int((tdf.n_groups == tdf.n_groups.max()).sum()),
        target_overlap=overlap,
        lines=lines_rows,
        inputs=[dict(key=p["entry"]["key"], file=p["entry"]["file"], md5=p["entry"]["md5"],
                     source=p["entry"]["source"], license=p["entry"]["license"], **p["info"])
                for p in per.values()],
    )
    (out / "manifest.json").write_text(json.dumps(manifest, indent=1))
    print(json.dumps({k: manifest[k] for k in ("n_genes", "n_targets_union", "targets_in_all_groups")}))


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--inspect", action="store_true")
    args = ap.parse_args()
    sources = json.loads((ROOT / "sources.json").read_text())
    assert_verified(sources)
    inspect(sources) if args.inspect else build(sources)


def assert_verified(sources: list[dict]) -> None:
    """Every input needs a verified md5: from fetch.py's log (local) or drive_fetch.py's sidecar."""
    raw = DATA / "raw"
    log_path = raw / "fetch_log.json"
    log = {r["key"]: r for r in json.loads(log_path.read_text())} if log_path.exists() else {}
    bad = []
    for e in sources:
        side = raw / (e["file"] + ".verified.json")
        if side.exists():
            ok = json.loads(side.read_text()).get("checksum") == e["md5"]
        else:
            ok = bool(log.get(e["key"], {}).get("md5_ok"))
        if not ok:
            bad.append(e["key"])
    assert not bad, f"md5 not verified for {bad}: run fetch.py or drive_fetch.py first"


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/vcc-code/sources.json
[
 {
  "key": "K562_ess",
  "file": "K562_essential_raw_bulk_01.h5ad",
  "url": "https://ndownloader.figshare.com/files/35773070",
  "bytes": 79766954,
  "md5": "8321d5d3ffc99db2a5c71edca4189735",
  "source": "Replogle et al. 2022, Figshare+ 10.25452/figshare.plus.20029387",
  "license": "CC BY 4.0",
  "kind": "pseudobulk"
 },
 {
  "key": "K562_gw",
  "file": "K562_gwps_raw_bulk_01.h5ad",
  "url": "https://ndownloader.figshare.com/files/35774443",
  "bytes": 374587922,
  "md5": "4570b53c9d62ff6df281e622f0350060",
  "source": "Replogle et al. 2022, Figshare+ 10.25452/figshare.plus.20029387",
  "license": "CC BY 4.0",
  "kind": "pseudobulk"
 },
 {
  "key": "RPE1",
  "file": "rpe1_raw_bulk_01.h5ad",
  "url": "https://ndownloader.figshare.com/files/35775581",
  "bytes": 95350546,
  "md5": "74765fa87635467a869ea972356ae0e7",
  "source": "Replogle et al. 2022, Figshare+ 10.25452/figshare.plus.20029387",
  "license": "CC BY 4.0",
  "kind": "pseudobulk"
 },
 {
  "key": "HepG2",
  "file": "NadigOConner2024_hepg2.h5ad",
  "url": "https://zenodo.org/api/records/13350497/files/NadigOConner2024_hepg2.h5ad/content",
  "bytes": 850590740,
  "md5": "af2be47f7477cf32fa6e4bec1c6a4868",
  "source": "Nadig, O'Conner et al. 2025 via scPerturb, Zenodo 10.5281/zenodo.13350497",
  "license": "cc-by-4.0",
  "kind": "singlecell"
 },
 {
  "key": "Jurkat",
  "file": "NadigOConner2024_jurkat.h5ad",
  "url": "https://zenodo.org/api/records/13350497/files/NadigOConner2024_jurkat.h5ad/content",
  "bytes": 1293665804,
  "md5": "d8b05d00bfbd686d37ffdd4293bc6c8c",
  "source": "Nadig, O'Conner et al. 2025 via scPerturb, Zenodo 10.5281/zenodo.13350497",
  "license": "cc-by-4.0",
  "kind": "singlecell"
 }
]

In [ ]:
%%writefile /content/vcc-code/catalog.json
{
 "frozen_utc": "2026-09-28T15:52:45Z",
 "tiers": {
  "mini": {
   "files": 5,
   "bytes": 2693961966
  },
  "sc_replogle": {
   "files": 2,
   "bytes": 19362753211
  },
  "sc_k562_gw": {
   "files": 1,
   "bytes": 65830941948
  },
  "cd4": {
   "files": 1,
   "bytes": 44566657140
  },
  "orion_hct116": {
   "files": 109,
   "bytes": 46576484789
  },
  "orion_hek293t": {
   "files": 223,
   "bytes": 79683336248
  }
 },
 "files": [
  {
   "tier": "mini",
   "file": "K562_essential_raw_bulk_01.h5ad",
   "url": "https://ndownloader.figshare.com/files/35773070",
   "bytes": 79766954,
   "algo": "md5",
   "checksum": "8321d5d3ffc99db2a5c71edca4189735",
   "source": "Replogle et al. 2022, Figshare+ 10.25452/figshare.plus.20029387",
   "license": "CC BY 4.0"
  },
  {
   "tier": "mini",
   "file": "K562_gwps_raw_bulk_01.h5ad",
   "url": "https://ndownloader.figshare.com/files/35774443",
   "bytes": 374587922,
   "algo": "md5",
   "checksum": "4570b53c9d62ff6df281e622f0350060",
   "source": "Replogle et al. 2022, Figshare+ 10.25452/figshare.plus.20029387",
   "license": "CC BY 4.0"
  },
  {
   "tier": "mini",
   "file": "rpe1_raw_bulk_01.h5ad",
   "url": "https://ndownloader.figshare.com/files/35775581",
   "bytes": 95350546,
   "algo": "md5",
   "checksum": "74765fa87635467a869ea972356ae0e7",
   "source": "Replogle et al. 2022, Figshare+ 10.25452/figshare.plus.20029387",
   "license": "CC BY 4.0"
  },
  {
   "tier": "sc_replogle",
   "file": "K562_essential_raw_singlecell_01.h5ad",
   "url": "https://ndownloader.figshare.com/files/35773219",
   "bytes": 10661879995,
   "algo": "md5",
   "checksum": "4f1122ce1c7f13299a68df6459a266d3",
   "source": "Replogle et al. 2022, Figshare+ 10.25452/figshare.plus.20029387",
   "license": "CC BY 4.0"
  },
  {
   "tier": "sc_replogle",
   "file": "rpe1_raw_singlecell_01.h5ad",
   "url": "https://ndownloader.figshare.com/files/35775606",
   "bytes": 8700873216,
   "algo": "md5",
   "checksum": "",
   "source": "Replogle et al. 2022, Figshare+ 10.25452/figshare.plus.20029387",
   "license": "CC BY 4.0"
  },
  {
   "tier": "sc_k562_gw",
   "file": "K562_gwps_raw_singlecell_01.h5ad",
   "url": "https://ndownloader.figshare.com/files/35775507",
   "bytes": 65830941948,
   "algo": "md5",
   "checksum": "",
   "source": "Replogle et al. 2022, Figshare+ 10.25452/figshare.plus.20029387",
   "license": "CC BY 4.0"
  },
  {
   "tier": "mini",
   "file": "NadigOConner2024_hepg2.h5ad",
   "url": "https://zenodo.org/api/records/13350497/files/NadigOConner2024_hepg2.h5ad/content",
   "bytes": 850590740,
   "algo": "md5",
   "checksum": "af2be47f7477cf32fa6e4bec1c6a4868",
   "source": "Nadig, O'Conner et al. 2025 via scPerturb, Zenodo 10.5281/zenodo.13350497",
   "license": "cc-by-4.0"
  },
  {
   "tier": "mini",
   "file": "NadigOConner2024_jurkat.h5ad",
   "url": "https://zenodo.org/api/records/13350497/files/NadigOConner2024_jurkat.h5ad/content",
   "bytes": 1293665804,
   "algo": "md5",
   "checksum": "d8b05d00bfbd686d37ffdd4293bc6c8c",
   "source": "Nadig, O'Conner et al. 2025 via scPerturb, Zenodo 10.5281/zenodo.13350497",
   "license": "cc-by-4.0"
  },
  {
   "tier": "cd4",
   "file": "GWCD4i.pseudobulk_merged.h5ad",
   "url": "https://genome-scale-tcell-perturb-seq.s3.amazonaws.com/marson2025_data/GWCD4i.pseudobulk_merged.h5ad",
   "bytes": 44566657140,
   "algo": "size_only",
   "checksum": null,
   "source": "Zhu, Dann, Marson et al. 2025, GEO GSE314342, public S3 bucket",
   "license": "not stated by the source: check before any use beyond research"
  },
  {
   "tier": "orion_hct116",
   "file": "HCT116_Batch1.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HCT116_Batch1.parquet",
   "bytes": 327620255,
   "algo": "sha256",
   "checksum": "a7a6918ecda30343e3f9262bcfb654bf077170d2e99c7bcd0c8b9f963764f13b",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hct116",
   "file": "HCT116_Batch10.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HCT116_Batch10.parquet",
   "bytes": 514603653,
   "algo": "sha256",
   "checksum": "c1179245d0a52e67565a7b1828a7e443c56eb164882a569de94f00e133891c58",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hct116",
   "file": "HCT116_Batch100.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HCT116_Batch100.parquet",
   "bytes": 465594768,
   "algo": "sha256",
   "checksum": "850472e07b7298c03c407cf71bd8795fca0ba4422b52ca4cd2af4e4026334cdc",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hct116",
   "file": "HCT116_Batch101.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HCT116_Batch101.parquet",
   "bytes": 501782869,
   "algo": "sha256",
   "checksum": "ed7e5ed1fd3787a11ab3f9703675eeee988fbbe13aa2f7368eef0a0262a0497f",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hct116",
   "file": "HCT116_Batch102.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HCT116_Batch102.parquet",
   "bytes": 456106566,
   "algo": "sha256",
   "checksum": "2ed34653202653783556f000dc81d2eff5b1f0a3f2a15786660234cc500ac352",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hct116",
   "file": "HCT116_Batch103.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HCT116_Batch103.parquet",
   "bytes": 457023750,
   "algo": "sha256",
   "checksum": "5c6af410396b85f81c1aeae7a5f8218365643dffc0a41c35897677429ce900d3",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hct116",
   "file": "HCT116_Batch104.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HCT116_Batch104.parquet",
   "bytes": 454227564,
   "algo": "sha256",
   "checksum": "7854cbce8d774a60481f9171316fc3860ea0767ceda83681c0f899fd2527e6d1",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hct116",
   "file": "HCT116_Batch105.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HCT116_Batch105.parquet",
   "bytes": 368983326,
   "algo": "sha256",
   "checksum": "b3d2c4337db32742f2e654fa7b9f4f391a2a9acb958e4ca37d50a9066cffdc26",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hct116",
   "file": "HCT116_Batch106.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HCT116_Batch106.parquet",
   "bytes": 224743692,
   "algo": "sha256",
   "checksum": "952eb87ea39490b54adf3eabe049053f8a1d9c342f158c56675d38ec604814b5",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hct116",
   "file": "HCT116_Batch107.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HCT116_Batch107.parquet",
   "bytes": 340672624,
   "algo": "sha256",
   "checksum": "d989e8cf2f32a863fca8723970c4655bda42e77f6311172f1900887215599eb4",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hct116",
   "file": "HCT116_Batch108.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HCT116_Batch108.parquet",
   "bytes": 323441588,
   "algo": "sha256",
   "checksum": "dce5328dc3e026426d4dd9eb250b21ae678d1bd1882e68af881c77a81c85362a",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hct116",
   "file": "HCT116_Batch109.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HCT116_Batch109.parquet",
   "bytes": 442737032,
   "algo": "sha256",
   "checksum": "b0b14f28936db51ef4bb9e94451eb636c6313f0be94b7a24133b4c14d85650a5",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hct116",
   "file": "HCT116_Batch11.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HCT116_Batch11.parquet",
   "bytes": 377740578,
   "algo": "sha256",
   "checksum": "2210743c1ee3bef07974aa7e1550e40da29a4b0a392baee90cf0fb55cde6ae4c",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hct116",
   "file": "HCT116_Batch12.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HCT116_Batch12.parquet",
   "bytes": 494035915,
   "algo": "sha256",
   "checksum": "4b8cf6901f60bd842d682df42e6490dbe7497223ca074c6fd2c01aeea372f8b3",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hct116",
   "file": "HCT116_Batch13.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HCT116_Batch13.parquet",
   "bytes": 505260195,
   "algo": "sha256",
   "checksum": "5f04bfd4f817e82442d04f2ea4580883793a45010192c68bd6c7bfe38060b72b",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hct116",
   "file": "HCT116_Batch14.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HCT116_Batch14.parquet",
   "bytes": 533245989,
   "algo": "sha256",
   "checksum": "fa2aba678999c782c69ab115267709b41fa540fecf7fe8690640b4a79f358e21",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hct116",
   "file": "HCT116_Batch15.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HCT116_Batch15.parquet",
   "bytes": 534503978,
   "algo": "sha256",
   "checksum": "5c28685f758f119183ce46cbcee7801916271f2dde0a89d7c1ca87232e4f3c50",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hct116",
   "file": "HCT116_Batch16.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HCT116_Batch16.parquet",
   "bytes": 561301037,
   "algo": "sha256",
   "checksum": "0b0e011732ea9f9a5870676b8216fcccf7638f8d6d04ef162c4c7d51ab3204f2",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hct116",
   "file": "HCT116_Batch17.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HCT116_Batch17.parquet",
   "bytes": 539271755,
   "algo": "sha256",
   "checksum": "00812dc7799a30d82280f8278687063b33af69ebd8ca4d4fbc9b16f91d0281cb",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hct116",
   "file": "HCT116_Batch18.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HCT116_Batch18.parquet",
   "bytes": 308544721,
   "algo": "sha256",
   "checksum": "680025d08da66f2b57048bf285d4da00553db664abe79be54e0477acf72c5ba7",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hct116",
   "file": "HCT116_Batch19.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HCT116_Batch19.parquet",
   "bytes": 375068900,
   "algo": "sha256",
   "checksum": "bca2338f3b74c75bcca965796a838b4585ee2e267879226a3b55685203ceb729",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hct116",
   "file": "HCT116_Batch2.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HCT116_Batch2.parquet",
   "bytes": 333969909,
   "algo": "sha256",
   "checksum": "811ef0d06362caa61411a42db0fe0cdc91fe5dddcb53f5f0447a175df2df25aa",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hct116",
   "file": "HCT116_Batch20.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HCT116_Batch20.parquet",
   "bytes": 455387746,
   "algo": "sha256",
   "checksum": "03715c6763793bb89a40ac35dd63d0a8a2e48c31830c142f8a0bf9b1240c33a8",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hct116",
   "file": "HCT116_Batch21.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HCT116_Batch21.parquet",
   "bytes": 445301703,
   "algo": "sha256",
   "checksum": "94e52b051888b418e3ee2b2b08f39e453fd635bd40e72e06a63bc7447dbb1d1d",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hct116",
   "file": "HCT116_Batch22.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HCT116_Batch22.parquet",
   "bytes": 414725369,
   "algo": "sha256",
   "checksum": "30a753ce31cb525283264c2b377c3fdf26be5d0f4e005d1c1b546ab2e04081e8",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hct116",
   "file": "HCT116_Batch23.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HCT116_Batch23.parquet",
   "bytes": 411561712,
   "algo": "sha256",
   "checksum": "8eb2083a077018433900d6f8bf5a1c45bacb4649671c22dbb111670504be246f",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hct116",
   "file": "HCT116_Batch24.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HCT116_Batch24.parquet",
   "bytes": 414331314,
   "algo": "sha256",
   "checksum": "be6a160e8dfdc266caacd169a6a1ba936d3c4dea1598b7cbf7c90a0ac61b34b0",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hct116",
   "file": "HCT116_Batch25.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HCT116_Batch25.parquet",
   "bytes": 393477494,
   "algo": "sha256",
   "checksum": "2375c892da19f293073c6b2ffc923e64d0543215355c507ca7223a21c833f715",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hct116",
   "file": "HCT116_Batch26.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HCT116_Batch26.parquet",
   "bytes": 435883637,
   "algo": "sha256",
   "checksum": "d0bafe2edd7eaaa0ad4001e1f7ca2f87ff8520fd8226c29f1251d981d70b167b",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hct116",
   "file": "HCT116_Batch27.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HCT116_Batch27.parquet",
   "bytes": 448033719,
   "algo": "sha256",
   "checksum": "7525f4724dd7f212495f2420cdc82d9b8ef4538f310992d689ac8ccd1883d77e",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hct116",
   "file": "HCT116_Batch28.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HCT116_Batch28.parquet",
   "bytes": 433923072,
   "algo": "sha256",
   "checksum": "dd19ef6835e06710644d764a8d54df6e7215b33d830ec1354e63b84f4df52003",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hct116",
   "file": "HCT116_Batch29.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HCT116_Batch29.parquet",
   "bytes": 478050297,
   "algo": "sha256",
   "checksum": "4b0873b6467844f5d30763ec96ae17053c9e612d059d0d94d3e783a2c7b0e0fa",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hct116",
   "file": "HCT116_Batch3.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HCT116_Batch3.parquet",
   "bytes": 340165482,
   "algo": "sha256",
   "checksum": "a1edffd0423c73f2d28f8589b87e41b68414bb6a3c7ee87ecb48485d4bb12beb",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hct116",
   "file": "HCT116_Batch30.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HCT116_Batch30.parquet",
   "bytes": 474141877,
   "algo": "sha256",
   "checksum": "600c7198953ab31d46aec5346368dc518c53b49ac5a8f2754de993b107ed895c",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hct116",
   "file": "HCT116_Batch31.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HCT116_Batch31.parquet",
   "bytes": 429074452,
   "algo": "sha256",
   "checksum": "4cfa5a7d5f3ce0eaa93b2e7dd56e79b5e6176a1af8377dfb1a5a827e1ddeefb2",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hct116",
   "file": "HCT116_Batch32.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HCT116_Batch32.parquet",
   "bytes": 549931295,
   "algo": "sha256",
   "checksum": "77598e4974c263cce730a9f09d5bf115720621c996c2524d457c3f89657f96d9",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hct116",
   "file": "HCT116_Batch33.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HCT116_Batch33.parquet",
   "bytes": 464654297,
   "algo": "sha256",
   "checksum": "6d9a948559a41f8879e703c31aa5bb0bb5af9a4a666361f9fb50996d58b5509b",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hct116",
   "file": "HCT116_Batch34.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HCT116_Batch34.parquet",
   "bytes": 496894421,
   "algo": "sha256",
   "checksum": "6c850a8f272a8d55451d6fc593a6d178b6a9427d6f0192b0ef5445d7a810262e",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hct116",
   "file": "HCT116_Batch35.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HCT116_Batch35.parquet",
   "bytes": 423481573,
   "algo": "sha256",
   "checksum": "c41244fae526a5ccb6247bc9f83ec7635b899a5775b97994dae4c982c08ae178",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hct116",
   "file": "HCT116_Batch36.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HCT116_Batch36.parquet",
   "bytes": 476486998,
   "algo": "sha256",
   "checksum": "788689eb2c7c4245968a6ba99660aaed701ee1d5bef4114cc3264022d7914322",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hct116",
   "file": "HCT116_Batch37.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HCT116_Batch37.parquet",
   "bytes": 464350721,
   "algo": "sha256",
   "checksum": "4f43322ba9073fd8727447e330216f9d60cedf93427010f4cb02baf9726102c9",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hct116",
   "file": "HCT116_Batch38.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HCT116_Batch38.parquet",
   "bytes": 411129665,
   "algo": "sha256",
   "checksum": "3cffba55a40d1641c860e7695f96e07fbb89d6357274d3640b43dd6b8d539e1f",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hct116",
   "file": "HCT116_Batch39.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HCT116_Batch39.parquet",
   "bytes": 395509532,
   "algo": "sha256",
   "checksum": "32856943cc1873d871a7fd2e158c2be65b2254290d14d60f29cb177a91d0055b",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hct116",
   "file": "HCT116_Batch4.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HCT116_Batch4.parquet",
   "bytes": 328807320,
   "algo": "sha256",
   "checksum": "e46cb832a09cfb31d74f82a90e88295350a6ae2d2d17c8fdd2708924870394c7",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hct116",
   "file": "HCT116_Batch40.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HCT116_Batch40.parquet",
   "bytes": 466737149,
   "algo": "sha256",
   "checksum": "c2dad51d11e6ce829f0d6500b9cd9303d7447ae8ef92b4f5b1caddb27564a365",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hct116",
   "file": "HCT116_Batch41.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HCT116_Batch41.parquet",
   "bytes": 451027401,
   "algo": "sha256",
   "checksum": "c82dc468740afed180734be24585b28a2b81a8551b69395aaef428640f08f36d",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hct116",
   "file": "HCT116_Batch42.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HCT116_Batch42.parquet",
   "bytes": 487128675,
   "algo": "sha256",
   "checksum": "a2a93ffc571e69ca206e05becf14a2ea36958a41238bc46da5235da833f687ea",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hct116",
   "file": "HCT116_Batch43.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HCT116_Batch43.parquet",
   "bytes": 491169432,
   "algo": "sha256",
   "checksum": "5638b8e3aaeaf76c0b5b066bff7ede01dba5384aa1a5e8125548fb44699eec56",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hct116",
   "file": "HCT116_Batch44.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HCT116_Batch44.parquet",
   "bytes": 510881870,
   "algo": "sha256",
   "checksum": "f3ea09d4e679958bca24003bdf268bbfc2044f983d293966e1dada2cbbac4db4",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hct116",
   "file": "HCT116_Batch45.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HCT116_Batch45.parquet",
   "bytes": 502315815,
   "algo": "sha256",
   "checksum": "7f7a2d0e1eccc78558c16868577f085f7a3e6aef9713e53b0fcc0fe4eed9cd72",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hct116",
   "file": "HCT116_Batch46.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HCT116_Batch46.parquet",
   "bytes": 503553362,
   "algo": "sha256",
   "checksum": "5164edc71c023bf03770d1fc1964faf1ad2d30a614d9b75118b02d320cbb6c2e",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hct116",
   "file": "HCT116_Batch47.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HCT116_Batch47.parquet",
   "bytes": 471291245,
   "algo": "sha256",
   "checksum": "78bacc551f76b889c3d55b12e55887fd4a0c39fda4f587ebd354641fadd77f72",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hct116",
   "file": "HCT116_Batch48.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HCT116_Batch48.parquet",
   "bytes": 350390322,
   "algo": "sha256",
   "checksum": "0ef68e4e6e5f9321d2d3448d3755ec81cd720ccf147de0c550aab086965445f5",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hct116",
   "file": "HCT116_Batch49.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HCT116_Batch49.parquet",
   "bytes": 438891882,
   "algo": "sha256",
   "checksum": "fa982c122b79d4329fca03b85cd0c03798ae199942b43405cdbf2b8506554133",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hct116",
   "file": "HCT116_Batch5.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HCT116_Batch5.parquet",
   "bytes": 344895177,
   "algo": "sha256",
   "checksum": "34f4d3fe7e0b3a856b85a8113958c6f7e7375363cfaa5eed791d553ed19afe83",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hct116",
   "file": "HCT116_Batch50.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HCT116_Batch50.parquet",
   "bytes": 392451280,
   "algo": "sha256",
   "checksum": "0fb6f787b11e21fe263ec95efdc3f1067142f284e04d0cd64ab84edf2d147eea",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hct116",
   "file": "HCT116_Batch51.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HCT116_Batch51.parquet",
   "bytes": 377871842,
   "algo": "sha256",
   "checksum": "fb9ad8247f424c5b115498e3d86b416c9fcb7f6eb25542f24c1852838ef4266f",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hct116",
   "file": "HCT116_Batch52.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HCT116_Batch52.parquet",
   "bytes": 347296925,
   "algo": "sha256",
   "checksum": "fa1ced2a73d896cba829d00c0c8008c7b4bee3ddcd4504f267e9a77a00c4c9b1",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hct116",
   "file": "HCT116_Batch53.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HCT116_Batch53.parquet",
   "bytes": 401168887,
   "algo": "sha256",
   "checksum": "4920319fcda269ba9f82ca179256c22726b336f4d519c227f2457acd7902ae67",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hct116",
   "file": "HCT116_Batch54.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HCT116_Batch54.parquet",
   "bytes": 390358708,
   "algo": "sha256",
   "checksum": "d86b3440acff0f33759a59bcba5670d755b219277f07bc1d6a2063cf1f334055",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hct116",
   "file": "HCT116_Batch55.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HCT116_Batch55.parquet",
   "bytes": 302334567,
   "algo": "sha256",
   "checksum": "c87861aae62a8336f8636f2717f41630e2bb84b9a7519e4eee205d1de7b5cad6",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hct116",
   "file": "HCT116_Batch56.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HCT116_Batch56.parquet",
   "bytes": 366604784,
   "algo": "sha256",
   "checksum": "92115e922c1c7471b1c5bef648c165c048c2fa954d761b83fb646a3ecee2882c",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hct116",
   "file": "HCT116_Batch57.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HCT116_Batch57.parquet",
   "bytes": 347679668,
   "algo": "sha256",
   "checksum": "19a0c80733955427e06c8adae34bcbc08404d0508755751f7b65aace17bbe7f0",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hct116",
   "file": "HCT116_Batch58.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HCT116_Batch58.parquet",
   "bytes": 364858383,
   "algo": "sha256",
   "checksum": "471e70397002c7f456dae22a10177f7e1ad7b8fcb768e34ea3e7840da583b6f3",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hct116",
   "file": "HCT116_Batch59.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HCT116_Batch59.parquet",
   "bytes": 362912109,
   "algo": "sha256",
   "checksum": "bc84b81f1a79af4d372a0db83e6c3c9d5c8ae948aff0ca7a0df91baf8a878a1e",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hct116",
   "file": "HCT116_Batch6.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HCT116_Batch6.parquet",
   "bytes": 293794566,
   "algo": "sha256",
   "checksum": "6dec049f9f2addd5fe2826a59b5a22951effbe4ca98715b5119aa9325a89ebf7",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hct116",
   "file": "HCT116_Batch60.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HCT116_Batch60.parquet",
   "bytes": 456773351,
   "algo": "sha256",
   "checksum": "3273675b4c48e1f35ca664db1751e3f47b98947cd8a4e3c3fccaa7ccc3e7c67a",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hct116",
   "file": "HCT116_Batch61.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HCT116_Batch61.parquet",
   "bytes": 336209250,
   "algo": "sha256",
   "checksum": "9348a4f59d31c0149d49a5b8603fe336a171e179aeb8538ac6157c87fb75b142",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hct116",
   "file": "HCT116_Batch62.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HCT116_Batch62.parquet",
   "bytes": 431972981,
   "algo": "sha256",
   "checksum": "a6f5d7536d1af59f55ac3e6f3e5bf7d927f71cf3bb7fbe18d8463dccb8c2b447",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hct116",
   "file": "HCT116_Batch63.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HCT116_Batch63.parquet",
   "bytes": 516665856,
   "algo": "sha256",
   "checksum": "471ca8325e56e38e9d6ad6efda102e895876081a7020843c5016fb425be24363",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hct116",
   "file": "HCT116_Batch64.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HCT116_Batch64.parquet",
   "bytes": 492706657,
   "algo": "sha256",
   "checksum": "866115b31a28d8029923cabe9ac948d51beaf3372cd58dc5e5050304a575d990",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hct116",
   "file": "HCT116_Batch65.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HCT116_Batch65.parquet",
   "bytes": 494961044,
   "algo": "sha256",
   "checksum": "4094f85430b62b03fd7c694641b75b5e24e45b31e9e5adea4c5de2bcc9c4defd",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hct116",
   "file": "HCT116_Batch66.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HCT116_Batch66.parquet",
   "bytes": 313113761,
   "algo": "sha256",
   "checksum": "420e14d2f8b14408724ed06f555fa6f3a60c1de7f3a2c31b4a9cd5d644c66843",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hct116",
   "file": "HCT116_Batch67.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HCT116_Batch67.parquet",
   "bytes": 374875932,
   "algo": "sha256",
   "checksum": "06d4e8185bf238d11ec96e52918afc2ed03d265c79d0a6b56354e472902af5da",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hct116",
   "file": "HCT116_Batch68.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HCT116_Batch68.parquet",
   "bytes": 366522196,
   "algo": "sha256",
   "checksum": "69a9a0a3f2dc6feba5b92efae1bc159c753e18ea31db19fe097b4de19141a903",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hct116",
   "file": "HCT116_Batch69.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HCT116_Batch69.parquet",
   "bytes": 364138851,
   "algo": "sha256",
   "checksum": "3a7baca72ea9b7ec319ec6429274c20bf137b57e0d04854c869229921ae387ea",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hct116",
   "file": "HCT116_Batch7.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HCT116_Batch7.parquet",
   "bytes": 279352582,
   "algo": "sha256",
   "checksum": "796583e02aba121f310e83c19328c77dfe9455470e4dc5409c8fce0608959eae",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hct116",
   "file": "HCT116_Batch70.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HCT116_Batch70.parquet",
   "bytes": 400812453,
   "algo": "sha256",
   "checksum": "2e3f42c517bfa7a74ca3776cab40b3ad8a041d7cad5bba62e81305b283c1c211",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hct116",
   "file": "HCT116_Batch71.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HCT116_Batch71.parquet",
   "bytes": 400376768,
   "algo": "sha256",
   "checksum": "e07d9182da8f02691d4b715e24a8e431391004bdcbc229fc5ca447fdfdc1a904",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hct116",
   "file": "HCT116_Batch72.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HCT116_Batch72.parquet",
   "bytes": 411210392,
   "algo": "sha256",
   "checksum": "62d2490fc3a8ef0fe3a313797f60e8f50d3de69c4222003f34c2b13b5db1d949",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hct116",
   "file": "HCT116_Batch73.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HCT116_Batch73.parquet",
   "bytes": 404445798,
   "algo": "sha256",
   "checksum": "95dc06981eb7a952e74cb7359a1f51cdab2af1324105b2fe14e05485376ba687",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hct116",
   "file": "HCT116_Batch74.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HCT116_Batch74.parquet",
   "bytes": 320488429,
   "algo": "sha256",
   "checksum": "ab7f91bd6eae8e6fb327ece125ec6a9e3b998344901c9800f1250898bbdd8251",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hct116",
   "file": "HCT116_Batch75.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HCT116_Batch75.parquet",
   "bytes": 357218088,
   "algo": "sha256",
   "checksum": "c487fce784099f45fbc7383c2eec949b78a5df065c5f74e4743c90a90d4e7205",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hct116",
   "file": "HCT116_Batch76.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HCT116_Batch76.parquet",
   "bytes": 340708502,
   "algo": "sha256",
   "checksum": "a06e1f208666bed753014f9937ed380cc0ca47af6b9fc7b73b48a392e88a1370",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hct116",
   "file": "HCT116_Batch77.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HCT116_Batch77.parquet",
   "bytes": 331032967,
   "algo": "sha256",
   "checksum": "e6a4ce471eeee1238ad16eee4e5d7477a7ec5812e8e36181ef77d141835b0930",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hct116",
   "file": "HCT116_Batch78.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HCT116_Batch78.parquet",
   "bytes": 507032635,
   "algo": "sha256",
   "checksum": "488655666ab254092eebc12c3da6e2d7349f2882e8d9a55e0282a51dc858e2bf",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hct116",
   "file": "HCT116_Batch79.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HCT116_Batch79.parquet",
   "bytes": 518793317,
   "algo": "sha256",
   "checksum": "9609ac4eede39acbdbede75f5491990324e293e949a443644d958c8f091c747e",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hct116",
   "file": "HCT116_Batch8.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HCT116_Batch8.parquet",
   "bytes": 251129066,
   "algo": "sha256",
   "checksum": "0fc937a8481fea0f4fcd74e6e168b7183dd0ffe61db9ff6c76015c5d1a303989",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hct116",
   "file": "HCT116_Batch80.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HCT116_Batch80.parquet",
   "bytes": 542426846,
   "algo": "sha256",
   "checksum": "df8eb3da9adb6a0d1ad6a0783690a1bec72d60065eca8b04b2eb2ac40f9338c3",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hct116",
   "file": "HCT116_Batch81.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HCT116_Batch81.parquet",
   "bytes": 561994868,
   "algo": "sha256",
   "checksum": "36234a759449e8f98a53119f60966ff8d87d86221c41552a8d888dcbd209c6c5",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hct116",
   "file": "HCT116_Batch82.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HCT116_Batch82.parquet",
   "bytes": 406989265,
   "algo": "sha256",
   "checksum": "8c7f57785b946e3a1cfc3ae6f956b9048652e59e644c9583f3e241944ba5def6",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hct116",
   "file": "HCT116_Batch83.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HCT116_Batch83.parquet",
   "bytes": 517452062,
   "algo": "sha256",
   "checksum": "94485e8b10138c2a3c65e37f331137987be1ef03bd101bbbb6c67e01e39edfa2",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hct116",
   "file": "HCT116_Batch84.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HCT116_Batch84.parquet",
   "bytes": 583092706,
   "algo": "sha256",
   "checksum": "fb8efa3cf0b34ebbfc73bf31ebd1fd36b9d6c3447303a8a92b470afb8e2ffe05",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hct116",
   "file": "HCT116_Batch85.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HCT116_Batch85.parquet",
   "bytes": 539397314,
   "algo": "sha256",
   "checksum": "3a06f3748e596468a638612e1201cd7c30d7890f2e4db3ed179e190e98ae9681",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hct116",
   "file": "HCT116_Batch86.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HCT116_Batch86.parquet",
   "bytes": 515371889,
   "algo": "sha256",
   "checksum": "f70aa8e21fca097f7adb58cd9e7f2ecd87825a3f1f9a6bc42f9feee37251b83e",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hct116",
   "file": "HCT116_Batch87.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HCT116_Batch87.parquet",
   "bytes": 402387805,
   "algo": "sha256",
   "checksum": "1a1711604f6713ce4faf289d254b266a213ac66e526a991ddd78facfe2602051",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hct116",
   "file": "HCT116_Batch88.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HCT116_Batch88.parquet",
   "bytes": 414097484,
   "algo": "sha256",
   "checksum": "1cdaa312aa4a9362eca8743b53fca9f0132aee35a565cc96895227bf86cb0fe3",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hct116",
   "file": "HCT116_Batch89.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HCT116_Batch89.parquet",
   "bytes": 485825476,
   "algo": "sha256",
   "checksum": "e0687d26b7ef8af5d57c04858588c2b66ce5f2f9acf089ab2b2b094d36e64ced",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hct116",
   "file": "HCT116_Batch9.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HCT116_Batch9.parquet",
   "bytes": 530300031,
   "algo": "sha256",
   "checksum": "c1e041e4a7b145bd54c53731abce1e88f0b00cda90ba8514903c58d689944e2b",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hct116",
   "file": "HCT116_Batch90.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HCT116_Batch90.parquet",
   "bytes": 442110829,
   "algo": "sha256",
   "checksum": "9ecd57e905406b6e803c02ee3f427f23b4bd5d78aaf4d4012d2b3bc9ad814b43",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hct116",
   "file": "HCT116_Batch91.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HCT116_Batch91.parquet",
   "bytes": 520063185,
   "algo": "sha256",
   "checksum": "54176c09e31ef44ece4adc31c5192015d3599a054e21a20fd7384afe75780af2",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hct116",
   "file": "HCT116_Batch92.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HCT116_Batch92.parquet",
   "bytes": 509018361,
   "algo": "sha256",
   "checksum": "b59cc18b8d5b99f770f2c5967da8e82c0ee221a7ffffc003a168046b617e1d5a",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hct116",
   "file": "HCT116_Batch93.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HCT116_Batch93.parquet",
   "bytes": 430312539,
   "algo": "sha256",
   "checksum": "2e687e58e9c1eb8c7cdcbeda39f4410b341caf4dadb6cf0777d97a0c71aea664",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hct116",
   "file": "HCT116_Batch94.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HCT116_Batch94.parquet",
   "bytes": 368767694,
   "algo": "sha256",
   "checksum": "252aa74b8d380dc10cb11bfafd64ffd0a311b25e018cdbfcc60118b36a22c59f",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hct116",
   "file": "HCT116_Batch95.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HCT116_Batch95.parquet",
   "bytes": 363806902,
   "algo": "sha256",
   "checksum": "f65f79209347544324c3257c73ba9618eb41b50597648f92490a56eb27493c23",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hct116",
   "file": "HCT116_Batch96.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HCT116_Batch96.parquet",
   "bytes": 553399436,
   "algo": "sha256",
   "checksum": "27fced46e7229c24c100df44b80fab9acf16798ceb55a46b46990248e502056f",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hct116",
   "file": "HCT116_Batch97.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HCT116_Batch97.parquet",
   "bytes": 407565665,
   "algo": "sha256",
   "checksum": "f1c70ec3b5021431c5d32adaac185de9f2fa6f252771ff387c289f13aa71ac35",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hct116",
   "file": "HCT116_Batch98.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HCT116_Batch98.parquet",
   "bytes": 399555923,
   "algo": "sha256",
   "checksum": "6fcf3f12e47f1252f0d3a3088f9a334b1a0f6a32ca287f61785eb02a1bc45325",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hct116",
   "file": "HCT116_Batch99.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HCT116_Batch99.parquet",
   "bytes": 420609224,
   "algo": "sha256",
   "checksum": "2cc6794e305b308082584c85a8e227c1ebe4288139daee70a26a82ea2f80ace6",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch1.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch1.parquet",
   "bytes": 370263016,
   "algo": "sha256",
   "checksum": "bc97ddfeef8df72fbd7aa03cf8e5fd1a6293e0d467735e5e59ce907df49af8d1",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch10.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch10.parquet",
   "bytes": 545528503,
   "algo": "sha256",
   "checksum": "abde0e8c418540100054059da4de3c06465646b70e333a2b1b147b86b331d803",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch100.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch100.parquet",
   "bytes": 266472114,
   "algo": "sha256",
   "checksum": "60aa194b6963c84b682dcd49d1d07ce532e1b4f2012e38a00f135bcbecde3083",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch101.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch101.parquet",
   "bytes": 359876662,
   "algo": "sha256",
   "checksum": "0482ced5f548b0eda597bbafa308645cf7c7748da35d837f9044553cd930e0e5",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch102.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch102.parquet",
   "bytes": 339333100,
   "algo": "sha256",
   "checksum": "dee991de6c3586d3617e95af3a08a4b1f035c4264abec08a37d6fadc1ab467e4",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch103.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch103.parquet",
   "bytes": 391537576,
   "algo": "sha256",
   "checksum": "385779c237b70b4caf7a82ce875fa615c0250b8ed7653f571a990192469a1242",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch104.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch104.parquet",
   "bytes": 374673812,
   "algo": "sha256",
   "checksum": "08d9a4ca1e7205661aa999850eefc23e0292a47d7481289aaecb21187b3b6f8d",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch105.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch105.parquet",
   "bytes": 358392498,
   "algo": "sha256",
   "checksum": "c0a009f916c229b23b36094aa7d3f4ac5d7617bd7e2ca7746939ecb78be9bfd1",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch106.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch106.parquet",
   "bytes": 349096936,
   "algo": "sha256",
   "checksum": "5a53af725d958f8a6fe8e59beace2888660b502b74a0b0d45db8c69a0d893256",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch107.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch107.parquet",
   "bytes": 309462637,
   "algo": "sha256",
   "checksum": "84d271e16c4fff8df0ce42d54b1cb0b8bdaa6e3769b8b60526066f5476638a2e",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch108.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch108.parquet",
   "bytes": 254112922,
   "algo": "sha256",
   "checksum": "fa909712ddddd6d1c040f3433b0013d8144523538c9693ce667f20a6ec63f0bc",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch109.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch109.parquet",
   "bytes": 348655827,
   "algo": "sha256",
   "checksum": "175cebfed042488c54e48457e389a3765be3227c869f8e3cffaa9570438ab404",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch11.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch11.parquet",
   "bytes": 569729144,
   "algo": "sha256",
   "checksum": "f1efed5347ef9ab6ca0f40bd09fb3729dad2b0b87a447dfe3d58d5db27692d41",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch110.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch110.parquet",
   "bytes": 252939336,
   "algo": "sha256",
   "checksum": "f163ccd594a09af3fc84160793ff1ee366480fd03946f40bb5148fab327c6c6a",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch111.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch111.parquet",
   "bytes": 409929395,
   "algo": "sha256",
   "checksum": "4eb5ff011b2e62a3e678ec0163e1c936706134278a8ebecd8cc035b4dd67c5ba",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch112.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch112.parquet",
   "bytes": 435305419,
   "algo": "sha256",
   "checksum": "0a4ea840f93e3845258801d1a1e95cd5e4e66c23714a28cb5250c3911127fd48",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch113.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch113.parquet",
   "bytes": 339454023,
   "algo": "sha256",
   "checksum": "7f3b7a38f603b4a4488c56b4a364c48b939fcb0cf9b5767bf08ef498219600a8",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch114.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch114.parquet",
   "bytes": 304180475,
   "algo": "sha256",
   "checksum": "637d5b4d88adaa3ec37da8155e6c474f5c82a93ac079ac7d35b711c6b92bdffa",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch115.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch115.parquet",
   "bytes": 383494862,
   "algo": "sha256",
   "checksum": "1aa6abdfbd10aa36f3ec9f19793fb910b37892c45291336b7b32980f198186f9",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch116.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch116.parquet",
   "bytes": 387191613,
   "algo": "sha256",
   "checksum": "b2a1463930e3d7c69998a2b9d51898a142e41a284efca45f017c629bdf3ec42f",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch117.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch117.parquet",
   "bytes": 385130484,
   "algo": "sha256",
   "checksum": "478c078fdffc3d6b8c76600b3e0f300bc3a1cf0f71c5790e23f792f868f7c07e",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch118.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch118.parquet",
   "bytes": 374070941,
   "algo": "sha256",
   "checksum": "12fec9872fc0d0c46865f695a75bdb0b86d6dbb6cd2286555984064a059f01a4",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch119.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch119.parquet",
   "bytes": 388635147,
   "algo": "sha256",
   "checksum": "fadc3230c55009eaaceaa785c6a2c580608cbfcd8e374e6718e484abebca0cbd",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch12.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch12.parquet",
   "bytes": 570862060,
   "algo": "sha256",
   "checksum": "b188223ce6a2d0114b5955fa2f6899573f7a485f817998104bdac313628c6a3c",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch120.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch120.parquet",
   "bytes": 359889064,
   "algo": "sha256",
   "checksum": "5ef2b0ce93117c680331a77f9ab93281efdea309c583340c240f9ba48d3564da",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch121.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch121.parquet",
   "bytes": 466309396,
   "algo": "sha256",
   "checksum": "e202d1052f1db591c6b15cb55e2224d4b2a97d9c8e24c29f98d1d87dd4af8da7",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch122.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch122.parquet",
   "bytes": 420980984,
   "algo": "sha256",
   "checksum": "b5b67ea8e954e0b96c808bbfab47b2c8c2012d081495696fc3b1bc4c5dd73c75",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch123.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch123.parquet",
   "bytes": 420139974,
   "algo": "sha256",
   "checksum": "7106c4b92dda7bb5b5e1a1bf0381c91bdb9e5fcd1a404b77d797e3a5ea7e56eb",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch124.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch124.parquet",
   "bytes": 396019392,
   "algo": "sha256",
   "checksum": "e92294656b17f8e6ece5cf0525c387f093b368e0018af2c69cc90a3c039332bf",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch125.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch125.parquet",
   "bytes": 334983816,
   "algo": "sha256",
   "checksum": "736521926711a6b2304528a7aaf3df0f37a5878f4a77d13e2b82b6cf7715ba28",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch126.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch126.parquet",
   "bytes": 403210098,
   "algo": "sha256",
   "checksum": "ff23d47781c586e4fd06aa93c96904c3dbf4a260e2ce0306a68aad5448a69589",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch127.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch127.parquet",
   "bytes": 361562600,
   "algo": "sha256",
   "checksum": "b0b66a7d1284f37653e810acff6137d941ddc2119bfe00322d6ac790fbd882ae",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch128.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch128.parquet",
   "bytes": 353722633,
   "algo": "sha256",
   "checksum": "79bc4b5b9ab797b5e2194a810ea5ee0345eaf02833ac79442ae18c60b050fad6",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch129.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch129.parquet",
   "bytes": 369384053,
   "algo": "sha256",
   "checksum": "a3b690b8030bf41b30571001222b1b4a68ac00a2b2ced2c1e69a85215a729196",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch13.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch13.parquet",
   "bytes": 546302382,
   "algo": "sha256",
   "checksum": "f8b6788b304296554661c4f83e757fe4ca2fd7b84bf7737d8d83e7e34fe6ca7e",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch130.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch130.parquet",
   "bytes": 397010067,
   "algo": "sha256",
   "checksum": "77dfb0d9a975c448346efaa158db6a0cbfdf25e52034192e44d78ecf2c48c9d5",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch131.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch131.parquet",
   "bytes": 444430123,
   "algo": "sha256",
   "checksum": "f202f4772cd3668c0a129c09b820642d76e6d60063b00d62ffa23702e517dbac",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch132.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch132.parquet",
   "bytes": 458839317,
   "algo": "sha256",
   "checksum": "86a73e88f518fc8a8a81d7e6b45f3c4c6cffe1483163a167bb45e293413e1dbc",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch133.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch133.parquet",
   "bytes": 291362664,
   "algo": "sha256",
   "checksum": "0480fa93ade63bd2f632851d5b926ae84031a1102e406c2df04421557e1b4ddc",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch134.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch134.parquet",
   "bytes": 209084229,
   "algo": "sha256",
   "checksum": "95237082067188523401a661008442801613040cdb36b3b300686307865bde30",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch135.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch135.parquet",
   "bytes": 462512247,
   "algo": "sha256",
   "checksum": "a7aab5f1ece481f34a1db779ffc9afaabfa7927241c49f64f598eae73a457a89",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch136.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch136.parquet",
   "bytes": 186697339,
   "algo": "sha256",
   "checksum": "2ffb4159cfbb7d75e58268394c047fb4b38c631e281fc80d1b540d5cee3a2a50",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch137.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch137.parquet",
   "bytes": 286300634,
   "algo": "sha256",
   "checksum": "f0849232e20debeced49b757343aefe66e4a9a3e337df8b9ebafb2e89b72f09a",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch138.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch138.parquet",
   "bytes": 284496362,
   "algo": "sha256",
   "checksum": "08009b5881017be21c547cf65a6579a9c49074b8a2973fafbce44673e3cf368a",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch139.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch139.parquet",
   "bytes": 282864583,
   "algo": "sha256",
   "checksum": "fe96c934ed5ec6dd53ec568ea7f4f1a7e0cf638aca21da3b653ecb6faebf9794",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch14.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch14.parquet",
   "bytes": 717945523,
   "algo": "sha256",
   "checksum": "dd1032fe6fbe4133b5915b615fb9b2424c76ac7fe6f2b7887a16ca6f93c9c40b",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch140.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch140.parquet",
   "bytes": 278645279,
   "algo": "sha256",
   "checksum": "2f310ada401405077e305c902c0987a5c7c53763b504097a1a14fcb1517bfffc",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch141.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch141.parquet",
   "bytes": 244018139,
   "algo": "sha256",
   "checksum": "2ba6e2956ab43e2b18bbf348aecbc4137c480361590176fe146c44a5c1449f8b",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch142.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch142.parquet",
   "bytes": 272113428,
   "algo": "sha256",
   "checksum": "2e76cb447a16160e31200b3bf71f6b4c1fc6757d18f477971fe16fb85f8f7c77",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch143.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch143.parquet",
   "bytes": 261306652,
   "algo": "sha256",
   "checksum": "5b084aabecc320e356e0a5e858499239b440acb804d0f758b8e37d5eae9c5eaf",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch144.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch144.parquet",
   "bytes": 254874028,
   "algo": "sha256",
   "checksum": "a2c6ea0da4e449c5bf117e6e441958b4d3ee13498e51695e1cbc300741396a23",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch145.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch145.parquet",
   "bytes": 306075352,
   "algo": "sha256",
   "checksum": "e625f2f19338520e23b2ca26748834ed58d3f8cab143f9f7559c539a35742a44",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch146.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch146.parquet",
   "bytes": 286241250,
   "algo": "sha256",
   "checksum": "37770ff4b256a8dded9d0aa3bb5521b9844b287ad9362f1a6218d463482edda0",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch147.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch147.parquet",
   "bytes": 288750143,
   "algo": "sha256",
   "checksum": "e5f80d1012a48198f7d732dfd72f13446029e50a29fb6b81b6b8d4769348e842",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch148.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch148.parquet",
   "bytes": 315618568,
   "algo": "sha256",
   "checksum": "2e45a6d954f6efc19ec6d3dea0fe883579df50aebbc41cc5fae20df390c1d6d0",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch149.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch149.parquet",
   "bytes": 261524106,
   "algo": "sha256",
   "checksum": "19922231cedffe4fae56abe77d56adeb877593b1fd2e2c08c0ffbc6733daf13f",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch15.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch15.parquet",
   "bytes": 373088279,
   "algo": "sha256",
   "checksum": "85701ced1d7514ae8cb60b223817125f6042cd24cff94edbc1d8c00796e3a88a",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch150.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch150.parquet",
   "bytes": 248445533,
   "algo": "sha256",
   "checksum": "0f86f6ad021731bc92f491d7e642ea75fd2299c6b5f47777006bf411d8198353",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch151.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch151.parquet",
   "bytes": 275535797,
   "algo": "sha256",
   "checksum": "55259fa8fc8b634b96fca1555402598cf687e754858c8057ee556a8d91d22511",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch152.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch152.parquet",
   "bytes": 256824912,
   "algo": "sha256",
   "checksum": "90c896f16b9b58dc1529d1cf906a70307d42a9c8c8974a832e3e765c6a0f52c0",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch153.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch153.parquet",
   "bytes": 342110391,
   "algo": "sha256",
   "checksum": "286be5a5dd859a546b3c77c937d6181525f5bf8f4df020515c1488b32cf82f92",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch154.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch154.parquet",
   "bytes": 322379974,
   "algo": "sha256",
   "checksum": "a9c2ed84afbe904212afdfa7efdb1b8cbeb3a5d0346696a682e3ac8319778910",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch155.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch155.parquet",
   "bytes": 318761060,
   "algo": "sha256",
   "checksum": "d3d297f8b165567581596975e0c278d6d12d960f1473463c0929ab4f68db513f",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch156.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch156.parquet",
   "bytes": 324825877,
   "algo": "sha256",
   "checksum": "a9131a405fbd4ad3a6d54240e34e612826d7dd15d25dbdb35e13214e06298407",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch157.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch157.parquet",
   "bytes": 357921143,
   "algo": "sha256",
   "checksum": "34f6597e1c07a4922e04f12e1b59537f66667c7e693be674b7f47afd50099540",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch158.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch158.parquet",
   "bytes": 353850731,
   "algo": "sha256",
   "checksum": "763395e5a7e3533befa40eb0c171e824e33e4f3d7102832d518a9786de178fad",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch159.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch159.parquet",
   "bytes": 266852833,
   "algo": "sha256",
   "checksum": "663ca83c6e6b48facfa9b6cb927247bd35992adef1212707359c53c6f6847b50",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch16.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch16.parquet",
   "bytes": 391884453,
   "algo": "sha256",
   "checksum": "e54a11fb656e7e85923ec555e216fbc2651d3ce10708804fcb0b6b1bbdc0afbc",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch160.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch160.parquet",
   "bytes": 256025390,
   "algo": "sha256",
   "checksum": "aacc7f07c3b5e92fd095c404a51b61aa6f3c3c3d5cd419e12800df436837c000",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch161.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch161.parquet",
   "bytes": 294940037,
   "algo": "sha256",
   "checksum": "14532ec4fff3f3bc7dbcfd15a1149fdc359da3d5954fc9273d97dbe0b0b4ab86",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch162.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch162.parquet",
   "bytes": 308443417,
   "algo": "sha256",
   "checksum": "62ea95dd65cea9ff0ced11ebc2381aa90bf9142983adc203d1435671ab5e795c",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch163.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch163.parquet",
   "bytes": 354849446,
   "algo": "sha256",
   "checksum": "d392c3856f82b6d77e6cf71f567eaa69ead6813d92af8d986f1338ce14e64253",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch164.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch164.parquet",
   "bytes": 337318639,
   "algo": "sha256",
   "checksum": "bcbdf58ad72fe83514f827d9f9c4533cf1bd03756ea210bb76bf271737593ae9",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch165.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch165.parquet",
   "bytes": 329266844,
   "algo": "sha256",
   "checksum": "13ca0e300ae0ae740fc15743c0101cba278e021e5f571adc21b40ca5ba1f87e0",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch166.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch166.parquet",
   "bytes": 310837482,
   "algo": "sha256",
   "checksum": "871057fa3efc79fb01d695ba45e6c9d85a76fe241641f3a998df354533a4e484",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch167.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch167.parquet",
   "bytes": 368909288,
   "algo": "sha256",
   "checksum": "aa4ccbff149e8beb5188d546d8a1ace4e136889c9d9ecd4fdf3df266bfd7e0e9",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch168.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch168.parquet",
   "bytes": 331234062,
   "algo": "sha256",
   "checksum": "c9dc73ed64cc4f03158a81bf60895051a090983a338bf97051f4a42f16bf5357",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch169.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch169.parquet",
   "bytes": 330910945,
   "algo": "sha256",
   "checksum": "4470b570793748aa5c3036f3f97cbc3c400ab5bac0278336995bd3b635802260",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch17.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch17.parquet",
   "bytes": 378654054,
   "algo": "sha256",
   "checksum": "78731649b8548a0d67682431f922a46732693c0be43c665f818d2ac0f19abc45",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch170.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch170.parquet",
   "bytes": 392695163,
   "algo": "sha256",
   "checksum": "87f89a90b7488dc404c5ed21c5bf3e6bbadca0a82a9bd13ae3882b4c973c0cb7",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch171.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch171.parquet",
   "bytes": 380924520,
   "algo": "sha256",
   "checksum": "b9cd7a5158a18cdde29b95f44107e36704e7fe128127c041731bdc97775c9370",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch172.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch172.parquet",
   "bytes": 393581804,
   "algo": "sha256",
   "checksum": "1d7bd96387facd6263a8d1a040614d553ff16051c80e09b2067c8ddc9c865c5c",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch173.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch173.parquet",
   "bytes": 377922625,
   "algo": "sha256",
   "checksum": "a0cd03d7c4bf5432544c7215de4707db4c1fb81f773e42f46e241399f5e90700",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch174.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch174.parquet",
   "bytes": 393071888,
   "algo": "sha256",
   "checksum": "d88ed4b4b004c6d0906f2b8d26bc8e7b0ab7f4694b98e84404c9dad2423a1805",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch175.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch175.parquet",
   "bytes": 407244176,
   "algo": "sha256",
   "checksum": "55d7bfd26a30913a8f4c4fc8663ef1d01ed34bf867562b715d63b8a59c166530",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch176.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch176.parquet",
   "bytes": 397038693,
   "algo": "sha256",
   "checksum": "bfe5678f6a375cd9c9b802c8171ab0a5f8014e5df551a33076a4e44e2dbfed27",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch177.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch177.parquet",
   "bytes": 383484025,
   "algo": "sha256",
   "checksum": "9d01248dcc285a02dbbb20cfeadbe33d442d4a102d3b38874dfe31aae27c2685",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch178.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch178.parquet",
   "bytes": 379342328,
   "algo": "sha256",
   "checksum": "14df5114a46a07f67ad05f5cc39f203bed61eccd71b5ca28baaf99057d57db1d",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch179.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch179.parquet",
   "bytes": 405201802,
   "algo": "sha256",
   "checksum": "4b8f9ddad0f0f4144197c9b3ada603d5d9e91cd6759547d10236a5bb2feac6fb",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch18.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch18.parquet",
   "bytes": 377238684,
   "algo": "sha256",
   "checksum": "5ede51776a7087655d555145d72347841e89400a81b39e7ca8c89f73ce6043ff",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch180.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch180.parquet",
   "bytes": 384599903,
   "algo": "sha256",
   "checksum": "f22907bfdaf89186da5eabc1419fb079873ab14dbb0bd5a3ab2d4381e8f9ddf5",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch181.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch181.parquet",
   "bytes": 385015462,
   "algo": "sha256",
   "checksum": "ccb7d04449f12582e0998ab8af1e0960bb3536632f5ed458855a771ebdee4c70",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch182.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch182.parquet",
   "bytes": 406615823,
   "algo": "sha256",
   "checksum": "399d956bedfa51498d767bbc712f04361be56b887a3d4e775059afd1a27bca13",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch183.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch183.parquet",
   "bytes": 353988856,
   "algo": "sha256",
   "checksum": "0b13cb19a9646a91f4a5f8cd492039220fb1e2cd0b32b6754588d2e773285c31",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch184.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch184.parquet",
   "bytes": 415371124,
   "algo": "sha256",
   "checksum": "a4f989c7321fcd98528c975f7add3a910598d23130216cf68d28139217b61e2e",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch185.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch185.parquet",
   "bytes": 354179915,
   "algo": "sha256",
   "checksum": "bb675d33fc1fa828c0febb0fc7ea17e2de879afb9ec7220e0dc6999bd94bd130",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch186.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch186.parquet",
   "bytes": 414947781,
   "algo": "sha256",
   "checksum": "f1ce173aa74e39bcbe20e04153e76998cf492ce645ffbd44faa5c6e380b9853b",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch187.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch187.parquet",
   "bytes": 394964737,
   "algo": "sha256",
   "checksum": "f45df70343f1d2bf69e55719386ea83e4417937ed0321b7ba7c27650cfc521b7",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch188.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch188.parquet",
   "bytes": 346956764,
   "algo": "sha256",
   "checksum": "06177ff13db8ad7bde9656684cf3bad7e092bde85ddefcccdaf6461e63954f2e",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch189.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch189.parquet",
   "bytes": 328905117,
   "algo": "sha256",
   "checksum": "b88525ddfc4e1d2a1e566da4b33367406650fdcfe7385119c1bf4c971593e5e3",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch19.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch19.parquet",
   "bytes": 402510591,
   "algo": "sha256",
   "checksum": "befe47a217c9c74e686e91cd69da15a165c637cefa4e30f725badb697745a5e0",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch190.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch190.parquet",
   "bytes": 315204570,
   "algo": "sha256",
   "checksum": "1746de91a8c8010bdc52c2335bd4ffa03700fc10d5ecde52199d92fcf8222ef2",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch191.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch191.parquet",
   "bytes": 343887999,
   "algo": "sha256",
   "checksum": "0f8606bc30b99ee9b186dbbd99c522f21b02995d6f896502e441eb969adff2fe",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch192.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch192.parquet",
   "bytes": 356815033,
   "algo": "sha256",
   "checksum": "ca2f4411b093eaa0cd55e14bdafe1f9d950ee39eae0b0af946ae8b6832dea400",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch193.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch193.parquet",
   "bytes": 331794450,
   "algo": "sha256",
   "checksum": "1aaa92bf4938de8ba2573320015d29341fec32f523f2052882830595995a831f",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch194.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch194.parquet",
   "bytes": 349720559,
   "algo": "sha256",
   "checksum": "8d35d357814fdce2e05ba06bbcdfffe16e5db3aa22f07a2eac8edabe12fb1642",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch195.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch195.parquet",
   "bytes": 356628551,
   "algo": "sha256",
   "checksum": "092255f39518128ac375c7786096f17b11fb8175a0fb3a96ecddbcd30575258c",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch196.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch196.parquet",
   "bytes": 347500652,
   "algo": "sha256",
   "checksum": "132808d940df174b9fc8726e3fbcb4c33481ec7e8770f1df700e8216f6d8cf88",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch197.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch197.parquet",
   "bytes": 357410515,
   "algo": "sha256",
   "checksum": "64a6db332974f50e8065e66f0b2502d6c4409a64b130c79ac214c01583678079",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch198.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch198.parquet",
   "bytes": 351444223,
   "algo": "sha256",
   "checksum": "d75767d31d960450b3e73eeb87ecfbaf1c7c765d546fbebe5d42c1ebb64e5759",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch199.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch199.parquet",
   "bytes": 433712233,
   "algo": "sha256",
   "checksum": "8450d10dc334688ac9d7ecf79c90eaae907b464a0f199c898ea2987e35ce5b52",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch2.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch2.parquet",
   "bytes": 507125705,
   "algo": "sha256",
   "checksum": "33e8cab52c207c40cb22336d576abf23d7f17f57db3dbf11d4735aac1d045900",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch20.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch20.parquet",
   "bytes": 408989845,
   "algo": "sha256",
   "checksum": "4e4b9a6acc71225f69988f1a27099235052a617804490c0cb0cbe371a64a4155",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch200.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch200.parquet",
   "bytes": 349009009,
   "algo": "sha256",
   "checksum": "c24c85e2a33e6f73e6ffbb21c52e13b6cf49fa5031b1e4f075a3ccc59086263b",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch201.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch201.parquet",
   "bytes": 291886448,
   "algo": "sha256",
   "checksum": "8c486ba5fba6900da8197a4eafd3be94875b0599b03cb42c143910183c95837e",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch202.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch202.parquet",
   "bytes": 310058726,
   "algo": "sha256",
   "checksum": "3d81d7e6d9454f1f7b7a359719b50aeeac7fca0305593a7507002a382ef7169c",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch203.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch203.parquet",
   "bytes": 330294678,
   "algo": "sha256",
   "checksum": "620e4aeefd94fbd88400d5890d3d9f913fa81e84520cf4db6039459265a8da61",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch204.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch204.parquet",
   "bytes": 329212465,
   "algo": "sha256",
   "checksum": "f31498698763d20020b1f66d1e6b038604b385000ba357d9e82a95af90fec601",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch205.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch205.parquet",
   "bytes": 282931309,
   "algo": "sha256",
   "checksum": "fc7c783e27956a834c8ca868b510256e276542414cbbdc02e45c64e637826dd1",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch206.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch206.parquet",
   "bytes": 289211529,
   "algo": "sha256",
   "checksum": "5a8df9a64068204f4deffddd6d036fbf902f6d70c5d1b0333254d0d1daef17e7",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch207.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch207.parquet",
   "bytes": 284302279,
   "algo": "sha256",
   "checksum": "dbcc5253e84e83ada62ac218750eb141b2b9cc577ff65f1f24a42ff653d4447f",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch208.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch208.parquet",
   "bytes": 239263356,
   "algo": "sha256",
   "checksum": "b52d5c49e4f5580809f8c796607d3731dee74384ae43574b4512ecf7546f76d8",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch209.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch209.parquet",
   "bytes": 381379743,
   "algo": "sha256",
   "checksum": "fa27847f2162213b3504f0ddf0f3f7d4558878cc84bfdc595da88563f79dcb9f",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch21.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch21.parquet",
   "bytes": 396957960,
   "algo": "sha256",
   "checksum": "041b2d576e9789d6db0f06aec9880bd96e499ea6a65b307028600862e6ca1362",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch210.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch210.parquet",
   "bytes": 379393484,
   "algo": "sha256",
   "checksum": "298b45622755be946a4a49b204b8ecf57094378cac84ecac7865cea3cc1e5c45",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch211.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch211.parquet",
   "bytes": 349289370,
   "algo": "sha256",
   "checksum": "6e0258bee94b1b307e91b6a9c6199205cc028bc0736023df23934da3bb20ba8c",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch212.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch212.parquet",
   "bytes": 362869082,
   "algo": "sha256",
   "checksum": "3c8ea615ebb59baca482816675880f34b76d937a4427a7278521cb9717ad0e9a",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch213.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch213.parquet",
   "bytes": 375788886,
   "algo": "sha256",
   "checksum": "07bbd756819f07696ad3fcdd755e93f482cfa1f8e3a964c0bfe1269f24db0e6d",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch214.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch214.parquet",
   "bytes": 373016508,
   "algo": "sha256",
   "checksum": "e687e368ace1ac1ccd07c05768825750a8f276d899f042ae222f28d22009ccb7",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch215.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch215.parquet",
   "bytes": 388018404,
   "algo": "sha256",
   "checksum": "c07ffae48072b939937d4fff84f19da4aec0844279ef5d9bd4873a6582c75841",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch216.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch216.parquet",
   "bytes": 370115188,
   "algo": "sha256",
   "checksum": "21b1ffa5d2342a205566cf961e5026b474c7316ffe45d5033a0ef00176ac3487",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch217.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch217.parquet",
   "bytes": 180241192,
   "algo": "sha256",
   "checksum": "1c213124b734acffed9de76589629865c9f4e5e89860ba9522daf51319e1adfa",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch218.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch218.parquet",
   "bytes": 130573306,
   "algo": "sha256",
   "checksum": "258f96dc44cdc5bb5cb9b597e03aabad9bd2e40d44acdc9245954cc7e2fdad17",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch219.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch219.parquet",
   "bytes": 115225118,
   "algo": "sha256",
   "checksum": "04e25f9533826cba97d89db4108bbaca476f2d956165a506a05fc51982419fc4",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch22.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch22.parquet",
   "bytes": 418252299,
   "algo": "sha256",
   "checksum": "01fe7455b7de09b21269867a979aca8a60d78c0e8af116ec6013bb7fbb30fcf2",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch220.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch220.parquet",
   "bytes": 254301962,
   "algo": "sha256",
   "checksum": "d31cff2e74730df473325bc6fd5e883ae6631557b72a34351f0205645702e644",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch221.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch221.parquet",
   "bytes": 214310576,
   "algo": "sha256",
   "checksum": "93e1afe12167770ad4b8e61d834d42ff84fe02d38d09ed4e53e767274e299c5b",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch222.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch222.parquet",
   "bytes": 142011153,
   "algo": "sha256",
   "checksum": "4a6a435c8e3dafadb3d7c2362c5bb22746c4a833be5b723fdd0e2b2573bff60f",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch223.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch223.parquet",
   "bytes": 259480348,
   "algo": "sha256",
   "checksum": "32f7d853bbb3088b8b787ca91c762eafb963a65b2175a7a36380706b368cc167",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch23.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch23.parquet",
   "bytes": 396720319,
   "algo": "sha256",
   "checksum": "ced4fea319a7883850c99b67acea54e93880577af1c8705575fd8cbd53b0ad0b",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch24.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch24.parquet",
   "bytes": 434853133,
   "algo": "sha256",
   "checksum": "eac2093d98b4d44552bda752c3cf48c0041f344f84b7d879a17ec1e5d4e05252",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch25.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch25.parquet",
   "bytes": 407324883,
   "algo": "sha256",
   "checksum": "70ec12374a403292fc07af068700c2b351eb2f1df2ef30639e52154082348a5e",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch26.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch26.parquet",
   "bytes": 397326793,
   "algo": "sha256",
   "checksum": "351ec4c7d0633a03684f9826fb9d5cb67db945688c49232222d30ebc0510f5aa",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch27.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch27.parquet",
   "bytes": 439542399,
   "algo": "sha256",
   "checksum": "9c9708e674427218f41df99f9ce27a20eede88998aec0a74ed47d8e38733d1a7",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch28.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch28.parquet",
   "bytes": 406057392,
   "algo": "sha256",
   "checksum": "96feb4e9203103734fa23c75e20d5eceb27b14ac813ddc6ade2628005f001f18",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch29.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch29.parquet",
   "bytes": 389635938,
   "algo": "sha256",
   "checksum": "22b9844329051615d5c51e3b39e062b1d1ee83231458d7ca0ab14abbb35ccd13",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch3.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch3.parquet",
   "bytes": 569125497,
   "algo": "sha256",
   "checksum": "0948fde4b60a92253980190fd9c1466bb4649ec559f83f020ad143d94f5079db",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch30.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch30.parquet",
   "bytes": 150030414,
   "algo": "sha256",
   "checksum": "b7cdae2413d5c66c65a82adcce399bf7026331e7f41fa0835b39ff88897a2513",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch31.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch31.parquet",
   "bytes": 360932409,
   "algo": "sha256",
   "checksum": "70cc9281e6cff01ad2544133bf9ccce53c700ab6b03468437163d91fe1783104",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch32.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch32.parquet",
   "bytes": 388283129,
   "algo": "sha256",
   "checksum": "d481cbb1fe7712c1c4a18cfe4d0dfdca1f15057be2c385a34ba830ec4f9aec85",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch33.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch33.parquet",
   "bytes": 334903983,
   "algo": "sha256",
   "checksum": "ab1dfd07ebd237176c4f40349ba0d6425f22fb33cfbb846c2254b2d95c5d4b16",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch34.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch34.parquet",
   "bytes": 396350413,
   "algo": "sha256",
   "checksum": "28b35ced708a8e2734e3856d3fc8821f4619fc4e81d68e382ba374fbf2831d6f",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch35.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch35.parquet",
   "bytes": 408567539,
   "algo": "sha256",
   "checksum": "e1a46ab725497a07488f43bcd163e12f9906ac6e79834fd2169d1b9d0b6f0a50",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch36.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch36.parquet",
   "bytes": 400062274,
   "algo": "sha256",
   "checksum": "a395c27aa22e41e8f56a8d5c6c540d656582c58b9985686a7f1e87332e98cdc5",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch37.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch37.parquet",
   "bytes": 398139406,
   "algo": "sha256",
   "checksum": "fe85486d773d8d2b8ea852c1c901b9e5f9d067adf0437880fe605f479739bec8",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch38.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch38.parquet",
   "bytes": 411661828,
   "algo": "sha256",
   "checksum": "f47e7f30bdcd9aec596627d520197324bf5d6ff3d613ce2962aec0a12a94e2bf",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch39.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch39.parquet",
   "bytes": 406103932,
   "algo": "sha256",
   "checksum": "956ceaf5cfb8f9c9625ce7ac94960ff4d3b130c270d9765ca6c8edef51e85462",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch4.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch4.parquet",
   "bytes": 539828343,
   "algo": "sha256",
   "checksum": "57f20b47fadfdbd38cce071d72b2fc89773d7f4b194822497003e4ac0d210117",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch40.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch40.parquet",
   "bytes": 407371912,
   "algo": "sha256",
   "checksum": "73b38d17f12aade4976e0711ab1a198a573b48955857c0e21485144f4215880a",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch41.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch41.parquet",
   "bytes": 402768254,
   "algo": "sha256",
   "checksum": "0567c6839aa12276976d2488f06afce7fc14382a3955dccf4bd3462321c0c484",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch42.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch42.parquet",
   "bytes": 378127915,
   "algo": "sha256",
   "checksum": "8657c49c7c39cc5f7dc8b1bedae3f1b5fafa22a73a5e793990d7a3d648867494",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch43.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch43.parquet",
   "bytes": 397562885,
   "algo": "sha256",
   "checksum": "fd30353f585980a12ffb4a312c0f1a1866739ddeaa6fdbdf7b6c1c19576b1344",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch44.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch44.parquet",
   "bytes": 365438712,
   "algo": "sha256",
   "checksum": "83a9c3524f3f84dc7f32dc15db35fd414c9ac47ec89f7614e29eba8e90354d5d",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch45.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch45.parquet",
   "bytes": 382652819,
   "algo": "sha256",
   "checksum": "bd6e43a814cc25c077ed8307923cc546cc3586e40c0db18113ef0fc6a83c92c4",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch46.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch46.parquet",
   "bytes": 406042887,
   "algo": "sha256",
   "checksum": "ba1021127262a772871983db10d35567b2336c2b7114ac8806970e2d5f5eab31",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch47.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch47.parquet",
   "bytes": 384652501,
   "algo": "sha256",
   "checksum": "2d13eb215053d3fdcdc00a37e6156b18357722e292034a4dfa4426fedb5df595",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch48.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch48.parquet",
   "bytes": 375350269,
   "algo": "sha256",
   "checksum": "24909e5a05aacd993ba5fce46e0a81dcecf46abad1e29ec8226852f7e4e2fe6e",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch49.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch49.parquet",
   "bytes": 384061596,
   "algo": "sha256",
   "checksum": "9a2cfbe49fd3f9a339ee3ae54d16256ac1419c393f7704191401746e2890940a",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch5.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch5.parquet",
   "bytes": 517417952,
   "algo": "sha256",
   "checksum": "a273770ad7ee13eb5f38bb6c267e91320eaa39c216ac1e8108c805ed1996679f",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch50.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch50.parquet",
   "bytes": 376695516,
   "algo": "sha256",
   "checksum": "4bcc53492c940803a9520e10fe509a0dd38d6d52e2be8769db1785f58606bd62",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch51.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch51.parquet",
   "bytes": 398071275,
   "algo": "sha256",
   "checksum": "a2d43b31c147826495b9c4fa7aee42ec488841e49ea702a94eeee1dacb2a4479",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch52.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch52.parquet",
   "bytes": 358599654,
   "algo": "sha256",
   "checksum": "9267f934ff5fc864b0b68bad9be9eb230a0995ec747f347d900d8de5aa357a4f",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch53.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch53.parquet",
   "bytes": 400042831,
   "algo": "sha256",
   "checksum": "7241060ee7ebfb102056b36e3e1702bd3f653f89f752e3483e87e8cfaf4180c5",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch54.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch54.parquet",
   "bytes": 394333933,
   "algo": "sha256",
   "checksum": "098c8d15851e649f6aeb25bce011c2e3dceec66184ed76a325ce4fc55bfc9a81",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch55.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch55.parquet",
   "bytes": 418795717,
   "algo": "sha256",
   "checksum": "1ba4e2c57137e5e045ba37af375ef2f8e7481eb8ae518da18136ec18a732d236",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch56.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch56.parquet",
   "bytes": 160248816,
   "algo": "sha256",
   "checksum": "ae9b771600d094af85988f449856fd1ec3812e0dec3489fdaff4687ab868a57a",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch57.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch57.parquet",
   "bytes": 317052703,
   "algo": "sha256",
   "checksum": "f93fb933b5db49a51100451656ed56f9f29cf34b5a061c4da568086e46281122",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch58.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch58.parquet",
   "bytes": 297082476,
   "algo": "sha256",
   "checksum": "d385fe5fc12a54fddee03a89bb1ac92b9f9ad81fffb0be10fe8f67a655d9bc4d",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch59.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch59.parquet",
   "bytes": 337773366,
   "algo": "sha256",
   "checksum": "5fc59c4b75ab23983538324e7a9fda9139a7a505bb2f5386369e82400414d633",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch6.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch6.parquet",
   "bytes": 559024020,
   "algo": "sha256",
   "checksum": "910442539ecc08a0d9e4de1bc64274a7a551777a3c9ce21d0cf2a6a76439e5e5",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch60.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch60.parquet",
   "bytes": 298673050,
   "algo": "sha256",
   "checksum": "793ea58932c8af5011b5e247dac4f6550188425b851d440f5ca00c5a11e706f2",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch61.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch61.parquet",
   "bytes": 281052508,
   "algo": "sha256",
   "checksum": "ac103995cca85bbcf588c3e3560eaaddce6c9bc4718857a0593380876b2b18e1",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch62.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch62.parquet",
   "bytes": 299923457,
   "algo": "sha256",
   "checksum": "b05ef811547f851d66456b47da0400d2053210a4c486efb6bfbbfa901897548a",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch63.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch63.parquet",
   "bytes": 321220113,
   "algo": "sha256",
   "checksum": "86ee2abba176945181ac9554a6fae78a77dda91c69224563f377604560809d4b",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch64.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch64.parquet",
   "bytes": 328336827,
   "algo": "sha256",
   "checksum": "efbd479c52d877cdb760d06c24973f1d08399a05a9174d30b0bc10eee946457e",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch65.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch65.parquet",
   "bytes": 282061711,
   "algo": "sha256",
   "checksum": "0787b57ee5b2fb8a2fcea5e594c06095107953fd90ec291e93819e4ebbe62380",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch66.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch66.parquet",
   "bytes": 297343448,
   "algo": "sha256",
   "checksum": "96d1a8666b485461953cad2ea8fd5442113f7e90def6104b974f65be5df9c759",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch67.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch67.parquet",
   "bytes": 286857819,
   "algo": "sha256",
   "checksum": "8908f61d8819bdeab26c9fbfb222011f7d260b60eb1e613332eb5e8532509088",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch68.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch68.parquet",
   "bytes": 339871772,
   "algo": "sha256",
   "checksum": "04c90988e829e8dba44f8599a8175c83c48f3c1bf34ad2b999938e4cdd8ce179",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch69.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch69.parquet",
   "bytes": 203017347,
   "algo": "sha256",
   "checksum": "ca75fa98cad958cbc12c0be59a72e13f0d4bdf43d4656c9610cb9c4226be6e86",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch7.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch7.parquet",
   "bytes": 595506252,
   "algo": "sha256",
   "checksum": "536a7dbe152bb259bf27902327ef8c994ced3bb9a29bef82ad4e23a7a08dc825",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch70.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch70.parquet",
   "bytes": 203507051,
   "algo": "sha256",
   "checksum": "c9dabca54f69f16297a4b16de09e992b83a1df20a78ba0dfbaf8fc8e427ce7c3",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch71.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch71.parquet",
   "bytes": 214837627,
   "algo": "sha256",
   "checksum": "409db7a9a91b5d7b6e4fb05d7d640563693aca151262a2acee096a9afc27cf92",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch72.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch72.parquet",
   "bytes": 6031551,
   "algo": "sha256",
   "checksum": "5b8d8e8534fb48e15638ea0b890c8eb256ca9570e564a71b3a878795fc732b23",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch73.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch73.parquet",
   "bytes": 205099432,
   "algo": "sha256",
   "checksum": "27741fbd03a4f58289c799fc01acb1203295e6428ff2d886e6b0045f60e4db04",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch74.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch74.parquet",
   "bytes": 74557163,
   "algo": "sha256",
   "checksum": "5a82a763fb52b54811bf353c307accd48394e335ad3ab96ba3ab2148fa8885d2",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch75.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch75.parquet",
   "bytes": 186315807,
   "algo": "sha256",
   "checksum": "c6307bfda85ec2e4886f56d05981fe8bb84abbb431cf29a12145583c50eb4806",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch76.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch76.parquet",
   "bytes": 223056421,
   "algo": "sha256",
   "checksum": "1b386e64f247c4b49d0d5bf35dfdcb55fea735fac4aec8c70d9b38109482cde1",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch77.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch77.parquet",
   "bytes": 233677016,
   "algo": "sha256",
   "checksum": "47965e22548bedd04cf2a1c531de83e97d313a1b93de1f4de7e9b9692b59a5db",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch78.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch78.parquet",
   "bytes": 204253852,
   "algo": "sha256",
   "checksum": "c9a50ee5034ffe865feb24597fb5f07d7fc58ae9ca46c7eb17f54b8d9cf860f8",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch79.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch79.parquet",
   "bytes": 203360382,
   "algo": "sha256",
   "checksum": "d103768a03d45c49e5853fc5fbb16d6b04204968aff315f175e734610977b13d",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch8.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch8.parquet",
   "bytes": 406513306,
   "algo": "sha256",
   "checksum": "e8e776ec2b7c95fa07a61f514bd773217e2cfa4388d55b063f3826ae27a4e204",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch80.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch80.parquet",
   "bytes": 473457791,
   "algo": "sha256",
   "checksum": "7e6ce39d4b9b2aa718a13a7184126f8026b6f16e92fcdb46244a0edb5579fa22",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch81.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch81.parquet",
   "bytes": 439588465,
   "algo": "sha256",
   "checksum": "74f130c306baa6638b9ae854ff78b43b595dc4bfcf5dd7c27237b86e37c809b6",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch82.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch82.parquet",
   "bytes": 465164142,
   "algo": "sha256",
   "checksum": "2e88be728f278071cc4292c40df239fb9825fac9826f248b248b2a750aca18eb",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch83.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch83.parquet",
   "bytes": 447520665,
   "algo": "sha256",
   "checksum": "c4bc0cbced2306f4a1937ed1bd2004e13ecca5bafc6edf3847f699661a709a4f",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch84.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch84.parquet",
   "bytes": 454877527,
   "algo": "sha256",
   "checksum": "487f591615099b120e13d846149fe0354395d1e2b446b118aaed47e6b8f154a0",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch85.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch85.parquet",
   "bytes": 452159040,
   "algo": "sha256",
   "checksum": "2b0aeb3880f593b214c4edea66afe18fbcc1053888fdf7016434a897103ee572",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch86.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch86.parquet",
   "bytes": 461401746,
   "algo": "sha256",
   "checksum": "e2aa0f29c2b3d9836d79e45c8de824893030e8cfb1e3ed9946fef819c02866e8",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch87.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch87.parquet",
   "bytes": 481213563,
   "algo": "sha256",
   "checksum": "5ed52b82cea4f28706f282a411ddb0d92251e103e7d431634a14070747220fd3",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch88.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch88.parquet",
   "bytes": 435366139,
   "algo": "sha256",
   "checksum": "a8b60c64d8c474dea15a6f4ffec3e52c7b99d26dc3f455199dd8af77c08cd68c",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch89.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch89.parquet",
   "bytes": 478554420,
   "algo": "sha256",
   "checksum": "f6d2efc2aef550e06f0adb9e494c22045e2b32eed67ea88620fda4beada420c8",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch9.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch9.parquet",
   "bytes": 561031571,
   "algo": "sha256",
   "checksum": "8070363ef7e98a3874dbea006f83ee6bba324ac9c01ebf6f9d02a0a1ab526fa2",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch90.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch90.parquet",
   "bytes": 458474335,
   "algo": "sha256",
   "checksum": "47f0d50740c9ab29d58fc2c98c43e5d2597bfbbf405fa1dd130ba80c44b2a22a",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch91.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch91.parquet",
   "bytes": 414649812,
   "algo": "sha256",
   "checksum": "37a977b715396e25d631e10807a32f4e59e5c26ce0c1adc106aea457a89c1609",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch92.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch92.parquet",
   "bytes": 453243803,
   "algo": "sha256",
   "checksum": "0e01685cc56a8319589a65d6127f5e0ea52747a80819a9cf09670e902d5d8bd6",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch93.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch93.parquet",
   "bytes": 494506509,
   "algo": "sha256",
   "checksum": "a8b33c4968ce6f9f4a9d5ef8442e35bc9a1818214b8be03d12ee62590ec8cf9a",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch94.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch94.parquet",
   "bytes": 492275465,
   "algo": "sha256",
   "checksum": "4a33b777edcba978ad26775412d98ef12ba8a8af140fb27e6fdf63750933a5cc",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch95.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch95.parquet",
   "bytes": 436821480,
   "algo": "sha256",
   "checksum": "e39ea935f7064795b551e09b8e6e8ad568bbe18f4ca4abc7693fdeacac6cfacc",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch96.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch96.parquet",
   "bytes": 454746312,
   "algo": "sha256",
   "checksum": "96731ba68988026584337a15e791178ad38e77f563782a44f23ff82fa9b3d4c8",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch97.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch97.parquet",
   "bytes": 403110760,
   "algo": "sha256",
   "checksum": "1a73b33b1c992af9daf4e7c8e1c7b5ae9c53d4560d7c2617ef2fba2f62a9a593",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch98.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch98.parquet",
   "bytes": 388349621,
   "algo": "sha256",
   "checksum": "3a5341cde8148c16c23a588c73050f47803acbcd9db8e850b1698c2f31638b65",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  },
  {
   "tier": "orion_hek293t",
   "file": "HEK293T_Batch99.parquet",
   "url": "https://huggingface.co/datasets/Xaira-Therapeutics/X-Atlas-Orion/resolve/53a5bc98d49247bcf967500292575c3d3602de31/data/HEK293T_Batch99.parquet",
   "bytes": 328287129,
   "algo": "sha256",
   "checksum": "f0297a6c80c627c4bc1dcc132f30f0f78fb9fc89b28cb383ceaebb0b31f7ec8a",
   "source": "X-Atlas/Orion (Xaira), Hugging Face Xaira-Therapeutics/X-Atlas-Orion at 53a5bc98d49247bcf967500292575c3d3602de31",
   "license": "cc-by-nc-sa-4.0"
  }
 ]
}

In [ ]:
%%writefile /content/vcc-code/data.py
"""Mini dataset in memory (on the GPU), fold construction and batches.

A sample is a pair (target t, query line q). Its inputs are what the final phase provides:
the control profile of q, and the effects of t already measured in *other* cell lines
(public sources). Its label is the effect of t measured in q.

Regimes on the held-out line H of a fold (targets are split by `fold` in targets.csv):
    CT  target never trained on, its source effects given as input   <- the final-phase case
    J   target never trained on, no source effects (gene descriptors only)
    C   target trained on in other lines, H new
Leakage rules (docs/GENERALIZZAZIONE.md of the team repo): the held-out line's perturbed
profiles never enter training, normalisation or descriptors; held-out targets' effects never
enter training or descriptors (readout SVD) in any line. tests/test_leakage.py checks them.

Sources are cell-line *groups*, not experiments (P3): the two K562 experiments enter as one
source, their effects averaged with weights on cells, so K562 is never counted twice and the
number of sources a query sees is at most (number of groups - 1).

Control features use basal_common.npy when present (P4): log1p(cpm) with the library summed
over the common genes only, so that the experiments' different gene panels do not shift the
controls of one line against another. The effects keep the full-library normalisation.
"""
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import torch


@dataclass
class Fold:
    held_group: str
    target_fold: int


class Mini:
    def __init__(self, root: Path, device: str = "cuda"):
        root = Path(root)
        self.device = device
        self.genes = pd.read_csv(root / "genes.csv").gene.to_numpy()
        self.lines = pd.read_csv(root / "lines.csv")
        self.keys = list(self.lines.line)
        self.group = dict(zip(self.lines.line, self.lines.group))
        self.groups = sorted(set(self.group.values()))
        self.lines_of = {g: [i for i, k in enumerate(self.keys) if self.group[k] == g] for g in self.groups}
        self.s_max = len(self.groups) - 1
        self.targets = pd.read_csv(root / "targets.csv")
        self.fold_of = dict(zip(self.targets.target, self.targets.fold))
        self.gidx = {g: i for i, g in enumerate(self.genes)}
        self.G = len(self.genes)
        common = root / "basal_common.npy"
        self.basal_file = common.name if common.exists() else "basal.npy"
        self.basal = torch.tensor(np.load(root / self.basal_file), device=device)      # [L, G]
        self.delta, self.row, self.meta = [], [], []
        for key in self.keys:
            z = np.load(root / f"effects_{key}.npz")
            self.delta.append(torch.tensor(z["delta"], device=device))
            self.row.append({t: i for i, t in enumerate(z["targets"])})
            n = np.nan_to_num(z["n_cells"], nan=100.0)
            kd = np.nan_to_num(z["kd"], nan=0.0)
            self.meta.append(dict(n=n, kd=kd))
        # Global similarity of control profiles, a transferable scalar per (source, query).
        b = self.basal - self.basal.mean(1, keepdim=True)
        b = b / b.norm(dim=1, keepdim=True)
        self.basal_sim = (b @ b.T).cpu().numpy()

    # ----------------------------------------------------------------- fold bookkeeping
    def split(self, fold: Fold, val_share: float = 0.1, seed: int = 0):
        """Training pairs, inner-validation pairs and held-out evaluation sets."""
        rng = np.random.default_rng(seed)
        train_lines = [i for i, k in enumerate(self.keys) if self.group[k] != fold.held_group]
        held_lines = [i for i, k in enumerate(self.keys) if self.group[k] == fold.held_group]
        test_t = {t for t, f in self.fold_of.items() if f == fold.target_fold}
        pool = sorted({t for li in train_lines for t in self.row[li]} - test_t)
        val_t = set(rng.choice(pool, size=int(len(pool) * val_share), replace=False))
        train_t = set(pool) - val_t
        pairs = lambda ts, lines: [(t, q) for q in lines for t in self.row[q] if t in ts]
        return dict(
            train_lines=train_lines, held_lines=held_lines,
            train=pairs(train_t, train_lines), val=pairs(val_t, train_lines),
            test_t=test_t, train_t=train_t, val_t=val_t,
        )

    def sources(self, t: str, q: int, allowed: list[int]) -> list[list[int]]:
        """Source groups for (t, q): per group of another line than q, the allowed lines that measured t."""
        gq = self.group[self.keys[q]]
        out = []
        for g in self.groups:
            if g == gq:
                continue
            ls = [s for s in self.lines_of[g] if s in allowed and t in self.row[s]]
            if ls:
                out.append(ls)
        return out

    def source_effect(self, t: str, ls: list[int]) -> torch.Tensor:
        """Cells-weighted mean effect of t over the experiments of one group."""
        n = np.array([self.meta[s]["n"][self.row[s][t]] for s in ls])
        d = torch.stack([self.delta[s][self.row[s][t]] for s in ls])
        w = torch.tensor(n / n.sum(), device=self.device, dtype=d.dtype)
        return (w[:, None] * d).sum(0)

    # ----------------------------------------------------------------- descriptors
    def readout_svd(self, train_lines: list[int], train_t: set, k: int = 64):
        """Gene descriptors from how each gene moves as a *readout* across training knockdowns.

        Rows: training targets in training lines only. The entry of a target's own gene in its
        own row (the on-target knockdown) is zeroed, so a gene's descriptor never contains its
        own perturbation: held-out targets and training targets are described the same way.
        Returns gene loadings [G, k] (V * S, scaled) and the basis V [G, k].
        """
        blocks = []
        for li in train_lines:
            rows = [(i, t) for t, i in self.row[li].items() if t in train_t]
            if not rows:
                continue
            M = self.delta[li][[i for i, _ in rows]].clone()
            for r, (_, t) in enumerate(rows):
                if t in self.gidx:
                    M[r, self.gidx[t]] = 0.0
            blocks.append(M)
        M = torch.cat(blocks)
        M = M - M.mean(0, keepdim=True)
        U, S, V = torch.svd_lowrank(M, q=k, niter=4)
        load = V * S / S[0]
        return load, V

    # ----------------------------------------------------------------- batches
    def batch(self, pairs, allowed_sources, gene_load, train=False, rng=None,
              p_nosrc=0.25, p_subset=0.3, src_override=None, max_src=None, line_centre=None):
        """Tensors for a list of (target, query-line) pairs.

        src_override='none' forces the J regime; max_src keeps a random subset of that many
        source groups (a sensitivity check, drawn with rng). line_centre {line: mean effect} adds
        "sc", the same slots filled with the cells-weighted centre of their lines (gamma = 1).
        """
        B, G, dev, S = len(pairs), self.G, self.device, self.s_max
        sd = torch.zeros(B, S, G, device=dev)
        sb = torch.zeros(B, S, G, device=dev)
        sm = torch.zeros(B, S, 3, device=dev)
        sc = torch.zeros(B, S, G, device=dev) if line_centre is not None else None
        mask = torch.zeros(B, S, dtype=torch.bool, device=dev)
        qb = torch.empty(B, G, device=dev)
        tf = torch.zeros(B, gene_load.shape[1] + 3, device=dev)
        y = torch.empty(B, G, device=dev)
        w = torch.empty(B, device=dev)
        gmask = torch.ones(B, G, dtype=torch.bool, device=dev)
        for b, (t, q) in enumerate(pairs):
            src = [] if src_override == "none" else self.sources(t, q, allowed_sources)
            if train and src:
                u = rng.random()
                if u < p_nosrc:
                    src = []
                elif u < p_nosrc + p_subset and len(src) > 1:
                    keep = rng.choice(len(src), size=rng.integers(1, len(src)), replace=False)
                    src = [src[i] for i in keep]
            if max_src is not None and len(src) > max_src:
                keep = rng.choice(len(src), size=max_src, replace=False)
                src = [src[i] for i in sorted(keep)]
            for j, ls in enumerate(src):
                n = np.array([self.meta[s]["n"][self.row[s][t]] for s in ls])
                kd = np.array([self.meta[s]["kd"][self.row[s][t]] for s in ls])
                sd[b, j] = self.source_effect(t, ls)
                sb[b, j] = self.basal[ls].mean(0)
                sm[b, j] = torch.tensor([np.log1p(n.sum()) / 7.0, float((kd * n).sum() / n.sum()),
                                         float(np.mean([self.basal_sim[s, q] for s in ls]))])
                mask[b, j] = True
                if sc is not None:
                    wn = n / n.sum()
                    sc[b, j] = sum(float(x) * line_centre[s] for x, s in zip(wn, ls))
            qb[b] = self.basal[q]
            y[b] = self.delta[q][self.row[q][t]]
            nq = float(self.meta[q]["n"][self.row[q][t]])
            w[b] = nq / (nq + 50.0)
            if t in self.gidx:
                g = self.gidx[t]
                gmask[b, g] = False                       # the scorer drops the own target gene
                tf[b, : gene_load.shape[1]] = gene_load[g]
                tf[b, -3] = 1.0
                tf[b, -2] = self.basal[q, g]
                tf[b, -1] = self.basal[allowed_sources, g].mean()
        out = dict(sd=sd, sb=sb, sm=sm, mask=mask, qb=qb, tf=tf, y=y, w=w, gmask=gmask)
        if sc is not None:
            out["sc"] = sc
        return out

In [ ]:
%%writefile /content/vcc-code/model.py
"""TransferNet: a gene-wise, source-set encoder for knockdown effects in an unseen cell line.

Prediction for target t in query line q, gene g:

    y_hat[g] = gain[g] * sum_s alpha[s, g] * delta_s[g]   +   U[g] . c
               '---------- transfer term ----------'       '- program term -'

- transfer term: a convex combination, per gene, of the effects of t already measured in the
  source lines. The attention weights alpha come from per-(source, gene) tokens built from
  transferable quantities only: the source effect, the gene's control expression in source and
  query, their difference, cells and on-target knockdown of the source measurement, and the
  global control similarity of source and query. With alpha uniform and gain constant this is a
  simple transfer (mean of the source effects times an amplitude). It is *not* the team's t22
  recipe, which also shrinks the effects, centres each source (gamma = 1), weights sources by
  reliability and adds a cis module.
- program term: a low-rank response whose coefficients c come from the target's descriptor and
  from the source effects projected on the programs. It is the only term left when no source
  measured t (regime J).

No parameter is indexed by gene or by cell line (P5). A gene is known to the model only through
its descriptor row in `gene_load` (how it moves as a readout across training knockdowns) and
through its control expression; the program loadings U are a function of the descriptor. The
parameter count does not depend on the number of genes, and the same weights run on any gene
axis for which the descriptors are computed the same way. `gene_res=True` adds back a learned
vector per gene, as an ablation only.
"""
import torch
import torch.nn as nn
import torch.nn.functional as F


def mlp(i, h, o, p=0.1):
    return nn.Sequential(nn.Linear(i, h), nn.GELU(), nn.Dropout(p), nn.Linear(h, o))


class TransferNet(nn.Module):
    def __init__(self, gene_load: torch.Tensor, d: int = 64, k_prog: int = 32,
                 gain_init: float = 0.5, gene_res: bool = False):
        super().__init__()
        G, kr = gene_load.shape
        assert k_prog <= kr
        self.register_buffer("gene_load", gene_load)                  # fixed readout descriptors
        self.gene_proj = mlp(kr, d, d)
        self.gene_res = nn.Embedding(G, d) if gene_res else None      # ablation only
        if gene_res:
            nn.init.zeros_(self.gene_res.weight)
        self.tok = nn.Linear(4 + 3, d)                                # per (source, gene) scalars + source meta
        self.tok_out = mlp(d, d, d)
        self.target = mlp(kr + 3, 2 * d, d)                           # target descriptor -> context vector
        self.query = mlp(d + 1 + d, d, d)                             # gene, query basal, target vector
        self.k = nn.Linear(d, d, bias=False)
        self.gain = mlp(2 * d + 1 + d, d, 1)
        nn.init.zeros_(self.gain[-1].weight)
        nn.init.constant_(self.gain[-1].bias, float(torch.log(torch.expm1(torch.tensor(gain_init)))))
        # Program loadings from the descriptor: starts as the first k_prog readout components.
        self.prog = nn.Linear(kr, k_prog, bias=False)
        with torch.no_grad():
            self.prog.weight.zero_()
            self.prog.weight[:, :k_prog] = torch.eye(k_prog)
        self.prog_res = mlp(d, d, k_prog)
        nn.init.zeros_(self.prog_res[-1].weight)
        nn.init.zeros_(self.prog_res[-1].bias)
        self.coef = mlp(d + k_prog + 1, 2 * d, k_prog)
        nn.init.zeros_(self.coef[-1].weight)
        nn.init.zeros_(self.coef[-1].bias)

    def forward(self, sd, sb, sm, mask, qb, tf, **_):
        B, S, G = sd.shape
        e = self.gene_proj(self.gene_load)                              # [G, d]
        if self.gene_res is not None:
            e = e + self.gene_res.weight
        U = self.prog(self.gene_load) + self.prog_res(e)                # [G, K]
        ctx = self.target(tf)                                           # [B, d]
        has = mask.any(1, keepdim=True).float()                         # [B, 1]

        x = torch.stack([sd, sb, qb[:, None] - sb, qb[:, None].expand_as(sd)], -1)   # [B,S,G,4]
        x = torch.cat([x, sm[:, :, None, :].expand(B, S, G, 3)], -1)
        h = self.tok_out(F.gelu(self.tok(x) + e))                        # [B,S,G,d]
        q = self.query(torch.cat([e.expand(B, G, -1), qb[..., None], ctx[:, None].expand(B, G, -1)], -1))
        logit = (self.k(h) * q[:, None]).sum(-1) / h.shape[-1] ** 0.5    # [B,S,G]
        logit = logit.masked_fill(~mask[:, :, None], -1e4)
        alpha = torch.softmax(logit, 1) * mask[:, :, None]
        transfer = (alpha * sd).sum(1)                                   # [B,G]
        pooled = (alpha[..., None] * h).sum(1)                           # [B,G,d]
        gain = F.softplus(self.gain(torch.cat([e.expand(B, G, -1), pooled, qb[..., None],
                                               ctx[:, None].expand(B, G, -1)], -1))).squeeze(-1)

        n_src = mask.sum(1, keepdim=True).clamp(min=1).float()
        src_mean = (sd * mask[:, :, None]).sum(1) / n_src                # [B,G]
        z = src_mean @ U / G ** 0.5                                      # projection on the programs
        c = self.coef(torch.cat([ctx, z, has], -1))                      # [B,K]
        program = c @ U.T
        return gain * transfer * has + program


def loss_fn(pred, y, w, gmask, lam_cos: float = 0.5):
    """Reliability-weighted MSE on the genes the scorer keeps, plus 1 - cosine (the PDS axis).

    The MSE is divided by the batch's mean squared effect (detached), so both terms are O(1):
    predicting zero costs 1 on the first term, as the scorer's `mse` normalises by effect size.
    """
    m = gmask.float()
    mse = ((pred - y) ** 2 * m).sum(1) / m.sum(1)
    scale = ((y ** 2 * m).sum(1) / m.sum(1)).mean().detach().clamp(min=1e-8)
    cos = F.cosine_similarity(pred * m, y * m, dim=1, eps=1e-8)
    return ((mse / scale + lam_cos * (1 - cos)) * w).sum() / w.sum()

In [ ]:
%%writefile /content/vcc-code/train.py
"""Train TransferNet on one fold and one seed; evaluate it against the references on the held-out line.

    python train.py --held HepG2 --fold 0 --seed 0 --out runs/hepg2_f0_s0

A fold holds out one cell-line group (all its experiments) and one fifth of the targets.
Selection (early stopping, reference amplitudes, the choice between the two transfer references)
uses only the inner-validation targets in the training lines. The held-out line is read once,
at the end.

Written to <out>/:
    results.json   configuration, counts, curve, summary metrics, paired bootstrap intervals
    per_target.npz per-target metrics of every method and regime, for aggregate.py

Methods compared on the same pairs:
    zero             no change
    train_mean       mean training effect, the same vector for every target
    oracle_mean      mean effect of the held-out line's evaluated targets (uses test labels:
                     the analogue of the scorer's 0 anchor, a reference and not a method)
    transfer_simple  mean over source groups of the source effects x amplitude fitted on validation
    transfer_gamma1  as above after subtracting each source line's mean training effect
                     (the team's gamma = 1 centring, computed on training targets only)
    replicate        K562 only: the other K562 experiment, same targets (a between-experiment
                     ceiling, lower than the scorer's within-experiment replicate)
    model            TransferNet
Neither transfer reference is the t22 recipe: no shrinkage, no reliability weights, no cis module.

Uncertainty (P1): for each regime, model minus each reference, per target; 2,000 paired bootstrap
resamples of the targets give a 95% interval of the mean difference. Seeds are combined in
aggregate.py.
"""
import argparse
import hashlib
import json
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F

from data import Fold, Mini
from model import TransferNet, loss_fn

ROOT = Path(__file__).resolve().parent
C_EVAL_SEED = 12345          # the C-regime subsample is the same for every training seed
N_BOOT = 2000


# --------------------------------------------------------------------------- metrics
def per_target(P: torch.Tensor, Y: torch.Tensor, gmask: torch.Tensor, pds_cols: torch.Tensor) -> dict:
    """Per-target vectors. pds_rank: cosine-distance rank of the own truth among the evaluated
    targets, with every evaluated target's gene removed (as the scorer removes panel genes);
    ties count half; 0 is perfect, 0.5 is chance."""
    m = gmask.float()
    mse = ((P - Y) ** 2 * m).sum(1) / m.sum(1)
    cos = (F.normalize(P * m, dim=1) * F.normalize(Y * m, dim=1)).sum(1)
    Pc, Yc = F.normalize(P[:, pds_cols], dim=1), F.normalize(Y[:, pds_cols], dim=1)
    D = 1 - Pc @ Yc.T
    d = D.diag()[:, None]
    rank = ((D < d).sum(1) + 0.5 * ((D == d).sum(1) - 1)) / max(D.shape[0] - 1, 1)
    k = min(100, Y.shape[1])
    top = (Y.abs() * m).topk(k, dim=1).indices
    ys, ps = Y.gather(1, top).sign(), P.gather(1, top).sign()
    return {"mse": mse, "cosine": cos, "pds_rank": rank.float(),
            "sign_top100": (ys == ps).float().mean(1), "down_top100": (ys < 0).float().mean(1)}


def summary(v: dict) -> dict:
    return {k: float(x.mean()) for k, x in v.items()} | {"n": int(len(v["mse"]))}


def paired_ci(model: np.ndarray, ref: np.ndarray, rng) -> dict:
    """Mean of (model - ref) per target, with a 95% paired bootstrap interval."""
    diff = model - ref
    idx = rng.integers(0, len(diff), size=(N_BOOT, len(diff)))
    boots = diff[idx].mean(1)
    return {"mean": float(diff.mean()), "lo": float(np.quantile(boots, 0.025)),
            "hi": float(np.quantile(boots, 0.975))}


# --------------------------------------------------------------------------- helpers
def run_batches(ds, pairs, allowed, load, fn, bs=32, **kw):
    outs = []
    for i in range(0, len(pairs), bs):
        outs.append(fn(ds.batch(pairs[i: i + bs], allowed, load, **kw)))
    return {k: torch.cat([o[k] for o in outs]) for k in outs[0]}


def transfer_refs(b: dict) -> dict:
    """Mean over the source groups present, raw and gamma = 1 centred."""
    m = b["mask"][:, :, None].float()
    n = m.sum(1).clamp(min=1)
    out = {"transfer_simple": (b["sd"] * m).sum(1) / n}
    if "sc" in b:
        out["transfer_gamma1"] = ((b["sd"] - b["sc"]) * m).sum(1) / n
    return out


def amplitude(T: torch.Tensor, Y: torch.Tensor) -> float:
    return float((T * Y).sum() / (T ** 2).sum().clamp(min=1e-12))


# --------------------------------------------------------------------------- main
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", default=str(ROOT / "mini"))
    ap.add_argument("--held", required=True)
    ap.add_argument("--fold", type=int, default=0)
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--out", required=True)
    ap.add_argument("--epochs", type=int, default=40)
    ap.add_argument("--patience", type=int, default=5)
    ap.add_argument("--bs", type=int, default=16)
    ap.add_argument("--lr", type=float, default=1e-3)
    ap.add_argument("--d", type=int, default=64)
    ap.add_argument("--k-prog", type=int, default=32)
    ap.add_argument("--k-readout", type=int, default=64)
    ap.add_argument("--lam-cos", type=float, default=0.5)
    ap.add_argument("--gene-res", action="store_true", help="ablation: learned vector per gene")
    ap.add_argument("--max-c-eval", type=int, default=1000)
    ap.add_argument("--max-train-pairs", type=int, default=0,
                    help="smoke runs only: keep at most this many training pairs of each kind (0 = all)")
    ap.add_argument("--device", default="cuda" if torch.cuda.is_available() else "cpu")
    args = ap.parse_args()
    out = Path(args.out)
    out.mkdir(parents=True, exist_ok=False)          # never overwrite a run
    torch.manual_seed(args.seed)
    rng = np.random.default_rng(args.seed)
    t0 = time.time()

    ds = Mini(Path(args.data), args.device)
    sp = ds.split(Fold(args.held, args.fold), seed=args.seed)
    tl = sp["train_lines"]
    load, _ = ds.readout_svd(tl, sp["train_t"], k=args.k_readout)

    # gamma = 1 centring: each training line's mean effect over *training* targets.
    line_mean = {li: torch.stack([ds.delta[li][i] for t, i in ds.row[li].items() if t in sp["train_t"]]).mean(0)
                 for li in tl}

    with_src = [p for p in sp["train"] if ds.sources(p[0], p[1], tl)]
    no_src = [p for p in sp["train"] if not ds.sources(p[0], p[1], tl)]
    val = [p for p in sp["val"] if ds.sources(p[0], p[1], tl)]
    if args.max_train_pairs:
        cut = lambda ps: [ps[i] for i in sorted(rng.choice(len(ps), min(len(ps), args.max_train_pairs), replace=False))] if ps else ps
        with_src, no_src, val = cut(with_src), cut(no_src), cut(val)

    model = TransferNet(load, d=args.d, k_prog=args.k_prog, gene_res=args.gene_res).to(args.device)
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    opt = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=1e-4)
    amp = dict(device_type="cuda", dtype=torch.bfloat16, enabled=args.device == "cuda")

    def val_loss():
        model.eval()
        tot, n = 0.0, 0
        with torch.no_grad(), torch.autocast(**amp):
            for i in range(0, len(val), 32):
                b = ds.batch(val[i: i + 32], tl, load)
                tot += float(loss_fn(model(**b).float(), b["y"], b["w"], b["gmask"], args.lam_cos)) * len(b["y"])
                n += len(b["y"])
        return tot / max(n, 1)

    curve, best, bad = [], float("inf"), 0
    ckpt = out / "best.pt"
    for ep in range(args.epochs):
        model.train()
        pick = rng.choice(len(no_src), size=min(len(with_src), len(no_src)), replace=False) if no_src else []
        epoch = with_src + [no_src[i] for i in pick]
        rng.shuffle(epoch)
        tr = 0.0
        for i in range(0, len(epoch), args.bs):
            b = ds.batch(epoch[i: i + args.bs], tl, load, train=True, rng=rng)
            with torch.autocast(**amp):
                loss = loss_fn(model(**b).float(), b["y"], b["w"], b["gmask"], args.lam_cos)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            tr += float(loss.detach()) * len(b["y"])
        vl = val_loss()
        curve.append(dict(epoch=ep, train=tr / len(epoch), val=vl, seconds=round(time.time() - t0)))
        print(curve[-1], flush=True)
        if vl < best - 1e-4:
            best, bad = vl, 0
            torch.save(model.state_dict(), ckpt)
        else:
            bad += 1
            if bad >= args.patience:
                break
    model.load_state_dict(torch.load(ckpt))
    model.eval()

    def refs_for(pairs, override=None, **kw):
        def f(b):
            with torch.no_grad(), torch.autocast(**amp):
                pred = model(**b).float()
            o = dict(model=pred, y=b["y"], g=b["gmask"], sd=b["sd"], mask=b["mask"])
            if "sc" in b:
                o["sc"] = b["sc"]
            return o
        centre = line_mean if override is None else None
        return run_batches(ds, pairs, tl, load, f, src_override=override, line_centre=centre, **kw)

    # Reference amplitudes and the choice between the two transfer references: validation only.
    rv = refs_for(val)
    Tv = transfer_refs(rv)
    Yv = rv["y"] * rv["g"]
    amps = {k: amplitude(T * rv["g"], Yv) for k, T in Tv.items()}
    val_mse = {k: float((((amps[k] * T - rv["y"]) ** 2) * rv["g"]).sum(1).div(rv["g"].sum(1)).mean())
               for k, T in Tv.items()}
    best_ref = min(val_mse, key=val_mse.get)

    train_mean = torch.stack([ds.delta[q][ds.row[q][t]] for t, q in sp["train"]]).mean(0)
    c_rng = np.random.default_rng(C_EVAL_SEED)
    s_rng = np.random.default_rng(C_EVAL_SEED + 1)
    boot_rng = np.random.default_rng(C_EVAL_SEED + 2)

    results, arrays = {}, {}
    for q in sp["held_lines"]:
        key = ds.keys[q]
        test = [(t, q) for t in ds.row[q] if t in sp["test_t"]]
        seen = sorted((t, q) for t in ds.row[q] if t in sp["train_t"] | sp["val_t"] and ds.sources(t, q, tl))
        if len(seen) > args.max_c_eval:
            seen = [seen[i] for i in sorted(c_rng.choice(len(seen), args.max_c_eval, replace=False))]
        ct = [p for p in test if ds.sources(p[0], p[1], tl)]
        regimes = {"CT": (ct, None, {}), "J": (test, "none", {}), "C": (seen, None, {})}
        if ds.s_max > 2:   # P3 sensitivity: at most as many sources as training ever showed
            regimes["CT_max2"] = (ct, None, {"max_src": ds.s_max - 1, "rng": s_rng})
        results[key] = {}
        for name, (pairs, override, kw) in regimes.items():
            if len(pairs) < 10:
                results[key][name] = {"skipped": f"{len(pairs)} pairs"}
                continue
            r = refs_for(pairs, override, **kw)
            Y, gm = r["y"], r["g"]
            cols = torch.ones(ds.G, dtype=torch.bool, device=ds.device)
            for t, _ in pairs:
                if t in ds.gidx:
                    cols[ds.gidx[t]] = False
            preds = {"zero": torch.zeros_like(Y), "train_mean": train_mean.expand_as(Y),
                     "oracle_mean": Y.mean(0, keepdim=True).expand_as(Y), "model": r["model"]}
            if override is None:
                for k, T in transfer_refs(r).items():
                    preds[k] = amps[k] * T
            pt = {k: per_target(v, Y, gm, cols) for k, v in preds.items()}
            if ds.group[key] == "K562":
                other = [s for s in sp["held_lines"] if s != q]
                both = [i for i, (t, _) in enumerate(pairs) if any(t in ds.row[s] for s in other)]
                if len(both) >= 10:
                    s = other[0]
                    idx = torch.tensor(both, device=ds.device)
                    R = torch.stack([ds.delta[s][ds.row[s][pairs[i][0]]] for i in both])
                    pt["replicate_same_targets"] = per_target(R, Y[idx], gm[idx], cols)
                    pt["model_same_targets"] = per_target(r["model"][idx], Y[idx], gm[idx], cols)
            res = {k: summary(v) for k, v in pt.items()}
            refs = [k for k in ("zero", "train_mean", "oracle_mean", "transfer_simple", "transfer_gamma1") if k in pt]
            res["model_minus_ref"] = {
                ref: {m: paired_ci(pt["model"][m].cpu().numpy(), pt[ref][m].cpu().numpy(), boot_rng)
                      for m in ("mse", "pds_rank", "cosine")} for ref in refs}
            results[key][name] = res
            for meth, v in pt.items():
                for m, x in v.items():
                    arrays[f"{key}|{name}|{meth}|{m}"] = x.cpu().numpy()
            arrays[f"{key}|{name}|targets"] = np.array([t for t, _ in pairs])

    record = dict(
        args=vars(args), seconds=round(time.time() - t0), basal_file=ds.basal_file,
        trainable_params=n_params,
        script_sha256={f: hashlib.sha256((ROOT / f).read_bytes()).hexdigest()
                       for f in ("train.py", "model.py", "data.py")},
        counts=dict(train_pairs=len(sp["train"]), with_sources=len(with_src), no_sources=len(no_src),
                    val_pairs_with_sources=len(val), test_targets=len(sp["test_t"])),
        reference_amplitudes=amps, reference_val_mse=val_mse, reference_selected_on_val=best_ref,
        curve=curve, results=results,
    )
    (out / "results.json").write_text(json.dumps(record, indent=1))
    np.savez_compressed(out / "per_target.npz", **arrays)
    for key, rr in results.items():
        for name, res in rr.items():
            if "skipped" in res:
                print(key, name, res)
                continue
            print(f"\n{key} {name}")
            for meth, mm in res.items():
                if meth == "model_minus_ref":
                    continue
                print(f"  {meth:24s} n={mm['n']:5d} mse={mm['mse']:.5f} cos={mm['cosine']:+.3f} "
                      f"pds_rank={mm['pds_rank']:.3f} sign={mm['sign_top100']:.3f}")
            ci = res["model_minus_ref"].get(best_ref)
            if ci:
                print(f"  model - {best_ref}: mse {ci['mse']['mean']:+.5f} [{ci['mse']['lo']:+.5f}, {ci['mse']['hi']:+.5f}]"
                      f"  pds_rank {ci['pds_rank']['mean']:+.3f} [{ci['pds_rank']['lo']:+.3f}, {ci['pds_rank']['hi']:+.3f}]")


if __name__ == "__main__":
    main()

In [ ]:
import hashlib
EXPECTED = {
 "drive_fetch.py": "cccfe74ce19f23a2c8310cfeaa84ce8a91c840eba4ece3477f29f4c6a9f6e6a9",
 "build_mini.py": "2f4f881cad4d4da1d1f900720ec45bfe3de847978a8b9fa1a44ab7bb7638b5a7",
 "sources.json": "875389621e13d4b2c3ff71ac08ad05a0616555d6b2dc4e34019d8684bb685dfe",
 "catalog.json": "0f5ca541c202ca8bba795c5c4063869fce2a306d9900fd7de64e00807d973572",
 "data.py": "3448cc37ea69ed4feeb5d217ee15c3ad3f7b519fab51796f3faf26d91cec46d5",
 "model.py": "f597a56875af853b04a9334a4b09d02474529499b02d90e65e10c4d21eaa6f94",
 "train.py": "438dee20ba42d87d62631b038a3611394ede5930cb81fb8a75de99842c8bdb9f"
}
for f, want in EXPECTED.items():
    got = hashlib.sha256(open(f"/content/vcc-code/{f}", encoding="utf-8").read().rstrip("\n").encode()).hexdigest()
    assert got == want, (f, got, want)
os.makedirs(f"{DRIVE_DIR}/code", exist_ok=True)
for f in EXPECTED:
    shutil.copyfile(f"/content/vcc-code/{f}", f"{DRIVE_DIR}/code/{f}")
print("code verified and copied to", f"{DRIVE_DIR}/code")

## Download verso Drive
Rilanciabile: salta ciÃ² che Ã¨ giÃ  verificato.

In [ ]:
!python /content/vcc-code/drive_fetch.py --catalog /content/vcc-code/catalog.json --dest "{DRIVE_DIR}" --tiers {" ".join(TIERS)} --limit {LIMIT}

## Ispezione dei file `mini` (sola lettura)

Copia i cinque file sul disco della VM e stampa colonne, etichette e un campione dei conteggi.
Non costruisce nulla: l'output serve a correggere `SPEC` in `build_mini.py` (punto P6 della revisione).
Il testo finisce anche in `DRIVE_DIR/logs/`.

In [ ]:
import glob, time
os.makedirs("/content/work/raw", exist_ok=True)
for p in glob.glob(f"{DRIVE_DIR}/raw/mini/*"):
    shutil.copyfile(p, "/content/work/raw/" + os.path.basename(p))
!pip -q install anndata
stamp = time.strftime("%Y%m%dT%H%M%SZ", time.gmtime())
!cd /content/vcc-code && VCC_MINI_DATA=/content/work python build_mini.py --inspect 2>&1 | tee "{DRIVE_DIR}/logs/inspect_{stamp}.txt"

## Costruzione del dataset (spenta)

Da accendere solo dopo aver controllato l'ispezione e corretto `SPEC`. Scrive in una cartella
nuova su Drive, `mini_<data>`, senza sovrascrivere quelle precedenti.

In [ ]:
BUILD = False
if BUILD:
    !cd /content/vcc-code && VCC_MINI_DATA=/content/work python build_mini.py
    stamp = time.strftime("%Y%m%dT%H%M%SZ", time.gmtime())
    shutil.copytree("/content/work/mini", f"{DRIVE_DIR}/mini_{stamp}")
    print("written", f"{DRIVE_DIR}/mini_{stamp}")

In [ ]:
drive.flush_and_unmount()  # scrive su Drive tutto ciÃ² che Ã¨ ancora in coda